# H121 vs legacy ASCAT O-F diagnostics

This notebook builds paper-style diagnostics from the monthly and full-period O-F summary files in `data/omf_compare_sums`.

The first-pass figure set emphasizes combined observation species groups:

- SMAP Tb species
- legacy ASCAT Metop-A/B/C species
- H SAF H121 ASCAT Metop-A/B/C species

The lower sections split the same metrics by individual species/platform so the combined diagnostics can be checked against the underlying behavior.

## Setup

In [ ]:
from __future__ import annotations

import calendar
import os
import struct
import sys
from pathlib import Path

import matplotlib as mpl
if "ipykernel" not in sys.modules:
    mpl.use("Agg")
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import BoundaryNorm, ListedColormap
from IPython.display import display
from netCDF4 import Dataset

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
except Exception:
    ccrs = None
    cfeature = None

ROOT = Path.cwd()
while ROOT.name != "geosldas-analysis" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if ROOT.name != "geosldas-analysis":
    ROOT = Path("/Users/amfox/Desktop/geosldas-analysis")

DATA_ROOT = ROOT / "data" / "omf_compare_sums"
OUT_DIR = ROOT / "projects" / "ascat_da" / "output" / "omf_h121_legacy_figures"
TILECOORD = ROOT / "projects" / "obs_scaling_params" / "test_data" / "inputs" / "OLv7_M36_MULTI_type_13_H121.ldas_tilecoord.bin"
OUT_DIR.mkdir(parents=True, exist_ok=True)

NMIN = 20
DPI = 180
ROLLING_WINDOW = 5
MAP_LAT_MIN = -60.0
LAND_FACE = "0.92"
ZERO_NEUTRAL_FRACTION = 0.02

RUN_LABELS = {
    "OL_vs_SMAPobs": "OL vs SMAP obs",
    "OL_vs_legacyobs": "OL vs legacy obs",
    "OL_vs_H121obs": "OL vs H121 obs",
    "DA_H121": "DA H121",
    "DA_legacy": "DA legacy",
    "DA_SMAP": "DA SMAP",
}

RUN_COLORS = {
    "DA_H121": "#1f77b4",
    "DA_legacy": "#d62728",
    "DA_SMAP": "#2ca02c",
}

PRIMARY_EXPERIMENTS = ("DA_H121", "DA_legacy")
ASSIMILATED_GROUPS_BY_EXPERIMENT = {
    "DA_H121": {"H121 ASCAT"},
    "DA_legacy": {"legacy ASCAT"},
    "DA_SMAP": {"SMAP"},
}

GROUP_COLORS = {
    "SMAP": "#ff7f0e",
    "legacy ASCAT": "#2ca02c",
    "H121 ASCAT": "#1f77b4",
}
GROUP_LINESTYLES = {
    "SMAP": "--",
    "legacy ASCAT": "-.",
    "H121 ASCAT": "-",
}

SPECIES_10 = [
    "SMAP_L1C_Tbh_A",
    "SMAP_L1C_Tbh_D",
    "SMAP_L1C_Tbv_A",
    "SMAP_L1C_Tbv_D",
    "ASCAT_META_SM",
    "ASCAT_METB_SM",
    "ASCAT_METC_SM",
    "ASCAT_HSAF_META_SM",
    "ASCAT_HSAF_METB_SM",
    "ASCAT_HSAF_METC_SM",
]

SPECIES_7 = SPECIES_10[:7]

EVALUATIONS = [
    {
        "group": "SMAP",
        "indices": (0, 1, 2, 3),
        "baseline": "OL_vs_SMAPobs",
        "experiments": ("DA_H121", "DA_legacy", "DA_SMAP"),
        "unit": "K",
    },
    {
        "group": "legacy ASCAT",
        "indices": (4, 5, 6),
        "baseline": "OL_vs_legacyobs",
        "experiments": ("DA_H121", "DA_legacy", "DA_SMAP"),
        "unit": "m3 m-3",
    },
    {
        "group": "H121 ASCAT",
        "indices": (7, 8, 9),
        "baseline": "OL_vs_H121obs",
        "experiments": ("DA_H121", "DA_legacy"),
        "unit": "m3 m-3",
    },
]

mpl.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": DPI,
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "legend.fontsize": 9,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

DATA_ROOT, OUT_DIR

## Loaders and helpers

In [ ]:
def _as_array(var):
    arr = np.array(var[:])
    if np.ma.isMaskedArray(arr):
        arr = arr.filled(np.nan)
    arr = arr.astype(float) if arr.dtype.kind in "fiu" else arr
    fill = getattr(var, "_FillValue", None)
    if fill is not None and arr.dtype.kind in "fiu":
        arr = np.where(arr == fill, np.nan, arr)
    return arr


def read_nc(path: Path) -> dict[str, np.ndarray]:
    with Dataset(path) as ds:
        out = {name: _as_array(var) for name, var in ds.variables.items()}
        out["dims"] = {name: len(dim) for name, dim in ds.dimensions.items()}
    return out


def yyyymm_to_timestamp(yyyymm):
    vals = np.asarray(yyyymm).astype(int)
    return pd.to_datetime([f"{v:06d}01" for v in vals], format="%Y%m%d")


def load_run(name: str) -> dict[str, dict[str, np.ndarray]]:
    base = DATA_ROOT / name
    monthly = read_nc(base / f"{name}_monthly_stats.nc4")
    temporal = read_nc(base / f"{name}_temporal_stats.nc4")
    monthly["months"] = yyyymm_to_timestamp(monthly["yyyymm"])
    return {"monthly": monthly, "temporal": temporal}


def _read_exact(fp, n_bytes: int) -> bytes:
    data = fp.read(n_bytes)
    if len(data) != n_bytes:
        raise EOFError(f"Expected {n_bytes} bytes, got {len(data)}")
    return data


def _read_record_tag(fp) -> int:
    return struct.unpack("<i", _read_exact(fp, 4))[0]


def read_tilecoord(path: Path) -> pd.DataFrame:
    int_fields = {"tile_id", "typ", "pfaf", "i_indg", "j_indg"}
    fields = [
        "tile_id",
        "typ",
        "pfaf",
        "com_lon",
        "com_lat",
        "min_lon",
        "max_lon",
        "min_lat",
        "max_lat",
        "i_indg",
        "j_indg",
        "frac_cell",
        "frac_pfaf",
        "area",
        "elev",
    ]
    out = {}
    with path.open("rb") as fp:
        tag = _read_record_tag(fp)
        if tag != 4:
            raise ValueError(f"{path} N_tile record tag is {tag}, expected 4")
        n_tile = struct.unpack("<i", _read_exact(fp, 4))[0]
        end_tag = _read_record_tag(fp)
        if end_tag != tag:
            raise ValueError(f"{path} N_tile record tags do not match")

        for field in fields:
            dtype = np.dtype("<i4") if field in int_fields else np.dtype("<f4")
            expected = n_tile * dtype.itemsize
            tag = _read_record_tag(fp)
            if tag != expected:
                raise ValueError(f"{path} field {field} tag is {tag}, expected {expected}")
            out[field] = np.frombuffer(_read_exact(fp, expected), dtype=dtype).copy()
            end_tag = _read_record_tag(fp)
            if end_tag != tag:
                raise ValueError(f"{path} field {field} record tags do not match")
    return pd.DataFrame(out)


def days_covered(months: pd.DatetimeIndex) -> int:
    return int(sum(calendar.monthrange(int(t.year), int(t.month))[1] for t in months))


def species_names_for_run(run_name: str) -> list[str]:
    n_species = runs[run_name]["monthly"]["OmF_stdv"].shape[1]
    return SPECIES_10[:n_species]


def has_species(data: dict[str, np.ndarray], indices) -> bool:
    return max(indices) < data["OmF_stdv"].shape[-1]


def weighted_group(values, n_data, indices, nmin=NMIN):
    if max(indices) >= values.shape[-1]:
        return np.full(values.shape[0], np.nan)
    idx = list(indices)
    vals = values[..., idx].astype(float)
    weights = n_data[..., idx].astype(float)
    valid = np.isfinite(vals) & np.isfinite(weights) & (weights >= nmin)
    num = np.where(valid, vals * weights, 0.0).sum(axis=-1)
    den = np.where(valid, weights, 0.0).sum(axis=-1)
    return np.divide(num, den, out=np.full(den.shape, np.nan), where=den > 0)


def grouped_count(n_data, indices):
    if max(indices) >= n_data.shape[-1]:
        return np.full(n_data.shape[0], np.nan)
    vals = n_data[..., list(indices)].astype(float)
    return np.where(np.isfinite(vals), vals, 0.0).sum(axis=-1)


def species_value(values, n_data, index, nmin=NMIN):
    if index >= values.shape[-1]:
        return np.full(values.shape[0], np.nan)
    vals = values[..., index].astype(float)
    weights = n_data[..., index].astype(float)
    return np.where(np.isfinite(vals) & np.isfinite(weights) & (weights >= nmin), vals, np.nan)


def percent_improvement(ol, da):
    ol = np.asarray(ol, dtype=float)
    da = np.asarray(da, dtype=float)
    return np.divide(100.0 * (ol - da), ol, out=np.full_like(ol, np.nan), where=np.isfinite(ol) & (np.abs(ol) > 0))


def percent_difference(exp, ol):
    exp = np.asarray(exp, dtype=float)
    ol = np.asarray(ol, dtype=float)
    return np.divide(100.0 * (exp - ol), ol, out=np.full_like(ol, np.nan), where=np.isfinite(ol) & (np.abs(ol) > 0))


def monthly_group_metric(cfg, run_name, metric="OmF_stdv"):
    data = runs[run_name]["monthly"]
    return weighted_group(data[metric], data["N_data"], cfg["indices"])


def temporal_group_metric(cfg, run_name, metric="OmF_stdv"):
    data = runs[run_name]["temporal"]
    return weighted_group(data[metric], data["N_data"], cfg["indices"])


def monthly_group_improvement(cfg, exp_run, metric="OmF_stdv"):
    ol = monthly_group_metric(cfg, cfg["baseline"], metric=metric)
    da = monthly_group_metric(cfg, exp_run, metric=metric)
    return percent_improvement(ol, da), ol, da


def temporal_group_improvement(cfg, exp_run, metric="OmF_stdv"):
    ol = temporal_group_metric(cfg, cfg["baseline"], metric=metric)
    da = temporal_group_metric(cfg, exp_run, metric=metric)
    return percent_improvement(ol, da), ol, da


def monthly_group_percent_difference(cfg, exp_run, metric="OmF_stdv"):
    ol = monthly_group_metric(cfg, cfg["baseline"], metric=metric)
    da = monthly_group_metric(cfg, exp_run, metric=metric)
    return percent_difference(da, ol), ol, da


def temporal_group_percent_difference(cfg, exp_run, metric="OmF_stdv"):
    ol = temporal_group_metric(cfg, cfg["baseline"], metric=metric)
    da = temporal_group_metric(cfg, exp_run, metric=metric)
    return percent_difference(da, ol), ol, da


def run_order_for_cfg(cfg):
    return (cfg["baseline"],) + tuple(cfg["experiments"])


def experiment_plot_order(cfg):
    preferred = ("DA_SMAP", "DA_legacy", "DA_H121")
    available = set(cfg["experiments"])
    return tuple(run for run in preferred if run in available)


def visible_run_order(cfg):
    preferred = (cfg["baseline"], "DA_SMAP", "DA_legacy", "DA_H121")
    available = set(run_order_for_cfg(cfg))
    return tuple(run for run in preferred if run in available)


def species_experiment_plot_order(row):
    preferred = ("DA_SMAP", "DA_legacy", "DA_H121")
    available = set(row["experiments"])
    return tuple(run for run in preferred if run in available)


def savefig(fig, stem: str):
    png = OUT_DIR / f"{stem}.png"
    fig.savefig(png, bbox_inches="tight")
    print(f"saved {png.relative_to(ROOT)}")
    display(fig)
    plt.close(fig)


def _panel_label_text(index: int) -> str:
    letters = []
    index = int(index)
    while True:
        letters.append(chr(ord("a") + index % 26))
        index = index // 26 - 1
        if index < 0:
            break
    return f"({''.join(reversed(letters))})"


def add_panel_labels(axes, *, x: float = 0.02, y: float = 0.98) -> None:
    for i, ax in enumerate(np.ravel(np.asarray(axes, dtype=object))):
        if ax is None or not ax.get_visible():
            continue
        ax.text(
            x, y, _panel_label_text(i),
            transform=ax.transAxes,
            ha="left", va="top",
            fontsize=10, fontweight="bold",
            bbox=dict(boxstyle="square,pad=0.15", facecolor="white", edgecolor="none", alpha=0.78),
            zorder=20,
        )


def _weighted_inputs(values, weights=None, latitudes=None):
    v = np.asarray(values, dtype=float)
    if weights is None and "tile_area" in globals() and np.asarray(tile_area).shape == v.shape:
        w = np.asarray(tile_area, dtype=float)
    elif weights is None:
        w = np.ones_like(v, dtype=float)
    else:
        w = np.asarray(weights, dtype=float)
    if w.shape != v.shape:
        raise ValueError(f"weights shape {w.shape} does not match values shape {v.shape}")
    ok = np.isfinite(v) & np.isfinite(w) & (w > 0)
    if latitudes is None and "lat" in globals() and np.asarray(lat).shape == v.shape:
        latitudes = lat
    if latitudes is not None:
        lats = np.asarray(latitudes, dtype=float)
        if lats.shape != v.shape:
            raise ValueError(f"latitudes shape {lats.shape} does not match values shape {v.shape}")
        ok &= np.isfinite(lats) & (lats >= MAP_LAT_MIN)
    return v, w, ok


def area_weighted_mean(values, weights=None, latitudes=None):
    v, w, ok = _weighted_inputs(values, weights=weights, latitudes=latitudes)
    if not ok.any():
        return np.nan
    return float(np.sum(v[ok] * w[ok]) / np.sum(w[ok]))


def area_weighted_std(values, weights=None, latitudes=None):
    v, w, ok = _weighted_inputs(values, weights=weights, latitudes=latitudes)
    if not ok.any():
        return np.nan
    mean = np.sum(v[ok] * w[ok]) / np.sum(w[ok])
    return float(np.sqrt(np.sum(w[ok] * (v[ok] - mean) ** 2) / np.sum(w[ok])))


def spatial_valid_count(values, latitudes=None):
    v, _, ok = _weighted_inputs(values, weights=np.ones_like(np.asarray(values, dtype=float)), latitudes=latitudes)
    return int(ok.sum())


def area_weighted_fraction(condition, valid_values, weights=None, latitudes=None):
    cond = np.asarray(condition, dtype=float)
    valid = np.asarray(valid_values, dtype=float)
    cond = np.where(np.isfinite(valid), cond, np.nan)
    return area_weighted_mean(cond, weights=weights, latitudes=latitudes)


def segmented_cmap_norm(cmap_name: str, vlim: float, n_bins: int = 12, neutral_fraction: float = ZERO_NEUTRAL_FRACTION):
    side_bins = max(int(n_bins) // 2, 1)
    color_bins = side_bins * 2
    neutral = max(float(vlim) * float(neutral_fraction), np.finfo(float).eps)
    neutral = min(neutral, float(vlim) * 0.5)
    neg_bounds = np.linspace(-vlim, -neutral, side_bins + 1)
    pos_bounds = np.linspace(neutral, vlim, side_bins + 1)
    bounds = np.r_[neg_bounds, pos_bounds]
    base = plt.get_cmap(cmap_name, color_bins)
    base_colors = base(np.linspace(0, 1, color_bins))
    colors = np.vstack([
        base_colors[:side_bins],
        np.array([[1.0, 1.0, 1.0, 1.0]]),
        base_colors[side_bins:],
    ])
    cmap = ListedColormap(colors, name=f"{cmap_name}_segmented_zero")
    norm = BoundaryNorm(bounds, cmap.N, clip=True)
    return cmap, norm


def set_diverging_colorbar_ticks(cb, vlim: float, fmt: str = "{:.0f}") -> None:
    ticks = np.linspace(-vlim, vlim, 5)
    cb.set_ticks(ticks)
    cb.ax.set_xticklabels([fmt.format(t) for t in ticks])


def segmented_log_cmap_norm(cmap_name: str, vmin: float, vmax: float, n_bins: int = 10):
    bounds = np.geomspace(float(vmin), float(vmax), int(n_bins) + 1)
    cmap = plt.get_cmap(cmap_name, int(n_bins))
    norm = BoundaryNorm(bounds, cmap.N, clip=True)
    return cmap, norm, bounds


def segmented_linear_cmap_norm(cmap_name: str, vmin: float, vmax: float, n_bins: int = 12):
    bounds = np.linspace(float(vmin), float(vmax), int(n_bins) + 1)
    cmap = plt.get_cmap(cmap_name, int(n_bins))
    norm = BoundaryNorm(bounds, cmap.N, clip=True)
    return cmap, norm, bounds


def format_time_axis(ax):
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=3))
    ax.grid(True, axis="y", color="0.86", linewidth=0.8)
    ax.grid(True, axis="x", color="0.93", linewidth=0.6)


def make_map_axis(fig, spec):
    if ccrs is None:
        return fig.add_subplot(spec)
    return fig.add_subplot(spec, projection=ccrs.Robinson())


MAP_EXTENT = (-180, 180, MAP_LAT_MIN, 85)


def decorate_map_axis(ax):
    if ccrs is None:
        ax.set_facecolor(LAND_FACE)
        ax.set_xlim(MAP_EXTENT[0], MAP_EXTENT[1])
        ax.set_ylim(MAP_EXTENT[2], MAP_EXTENT[3])
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        ax.grid(True, color="0.9", linewidth=0.6)
    else:
        ax.set_extent(MAP_EXTENT, crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.LAND, facecolor=LAND_FACE, edgecolor="none", zorder=0)
        ax.coastlines(linewidth=0.4, color="0.35", zorder=3)


def scatter_map(ax, lon, lat, values, *, cmap, norm=None, vmin=None, vmax=None, s=1.2):
    valid = np.isfinite(values) & np.isfinite(lon) & np.isfinite(lat) & (np.asarray(lat) >= MAP_LAT_MIN)
    kwargs = dict(c=values[valid], cmap=cmap, norm=norm, vmin=vmin, vmax=vmax, s=s, linewidths=0, rasterized=True, zorder=2)
    if ccrs is None:
        sc = ax.scatter(lon[valid], lat[valid], **kwargs)
    else:
        sc = ax.scatter(lon[valid], lat[valid], transform=ccrs.PlateCarree(), **kwargs)
    decorate_map_axis(ax)
    return sc



def latitude_band_summary(values, band_width: float = 5.0) -> pd.DataFrame:
    vals = np.asarray(values, dtype=float)
    edges = np.arange(MAP_LAT_MIN, MAP_EXTENT[3] + 0.1, band_width)
    rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        in_band = (lat >= lo) & ((lat < hi) if hi < edges[-1] else (lat <= hi))
        ok = in_band & np.isfinite(vals) & np.isfinite(tile_area) & (tile_area > 0)
        rows.append(
            {
                "lat_min": float(lo),
                "lat_max": float(hi),
                "lat_center": float((lo + hi) / 2.0),
                "value": area_weighted_mean(vals[ok], weights=tile_area[ok], latitudes=None) if ok.any() else np.nan,
                "n_valid_tiles": int(ok.sum()),
            }
        )
    return pd.DataFrame(rows)


def profile_y_from_lat(latitudes):
    latitudes = np.asarray(latitudes, dtype=float)
    if ccrs is None:
        return latitudes
    pts = ccrs.Robinson().transform_points(ccrs.PlateCarree(), np.zeros_like(latitudes), latitudes)
    return pts[:, 1]


## Read the O-F summaries

In [ ]:
run_names = ["OL_vs_SMAPobs", "OL_vs_legacyobs", "OL_vs_H121obs", "DA_H121", "DA_legacy", "DA_SMAP"]
runs = {name: load_run(name) for name in run_names}
tilecoord = read_tilecoord(TILECOORD)

n_tile = runs["DA_H121"]["temporal"]["OmF_stdv"].shape[0]
assert len(tilecoord) == n_tile, (len(tilecoord), n_tile)

months = runs["DA_H121"]["monthly"]["months"]
period_days = days_covered(months)
lon = tilecoord["com_lon"].to_numpy()
lat = tilecoord["com_lat"].to_numpy()
tile_area = tilecoord["area"].to_numpy()

inventory_rows = []
for name, run in runs.items():
    monthly = run["monthly"]
    temporal = run["temporal"]
    inventory_rows.append(
        {
            "run": name,
            "label": RUN_LABELS[name],
            "monthly_shape": monthly["OmF_stdv"].shape,
            "temporal_shape": temporal["OmF_stdv"].shape,
            "start": monthly["months"][0].strftime("%Y-%m"),
            "end": monthly["months"][-1].strftime("%Y-%m"),
            "species": ", ".join(species_names_for_run(name)),
        }
    )

inventory = pd.DataFrame(inventory_rows)
inventory

## Combined Species-Group Summary Table

In [ ]:
summary_rows = []
for cfg in EVALUATIONS:
    for exp in cfg["experiments"]:
        if not has_species(runs[exp]["monthly"], cfg["indices"]):
            continue
        month_imp, month_ol, month_da = monthly_group_improvement(cfg, exp)
        map_imp, map_ol, map_da = temporal_group_improvement(cfg, exp)
        month_diff, _, _ = monthly_group_percent_difference(cfg, exp)
        map_diff, _, _ = temporal_group_percent_difference(cfg, exp)
        summary_rows.append(
            {
                "group": cfg["group"],
                "experiment": exp,
                "experiment_label": RUN_LABELS[exp],
                "baseline": cfg["baseline"],
                "unit": cfg["unit"],
                "monthly_mean_ol_omf_stdv": np.nanmean(month_ol),
                "monthly_mean_da_omf_stdv": np.nanmean(month_da),
                "monthly_mean_pct_improvement": np.nanmean(month_imp),
                "monthly_mean_pct_difference_exp_minus_ol": np.nanmean(month_diff),
                "full_period_mean_ol_omf_stdv": area_weighted_mean(map_ol),
                "full_period_mean_da_omf_stdv": area_weighted_mean(map_da),
                "full_period_mean_pct_improvement": area_weighted_mean(map_imp),
                "full_period_median_pct_improvement": np.nanmedian(map_imp),
                "full_period_mean_pct_difference_exp_minus_ol": area_weighted_mean(map_diff),
                "full_period_median_pct_difference_exp_minus_ol": np.nanmedian(map_diff),
                "fraction_tiles_improved": area_weighted_fraction(map_imp > 0, map_imp),
                "n_valid_tiles": int(np.isfinite(map_imp).sum()),
            }
        )

summary = pd.DataFrame(summary_rows)
summary_path = OUT_DIR / "combined_omf_stdv_improvement_summary.csv"
summary.to_csv(summary_path, index=False)
summary_diff_path = OUT_DIR / "combined_omf_stdv_relative_difference_summary.csv"
summary.to_csv(summary_diff_path, index=False)
summary

## Fig. 2: experiment-first monthly O-F stddev relative difference

Monthly `(experiment - OL) / OL * 100` for each observation species group. More negative values mean the DA experiment has smaller O-F stddev than the matching OL/background run.


In [ ]:
fig, axes = plt.subplots(len(PRIMARY_EXPERIMENTS), 1, figsize=(13, 7.5), sharex=True, constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, exp in zip(axes, PRIMARY_EXPERIMENTS):
    all_vals = []
    for cfg in EVALUATIONS:
        if not has_species(runs[exp]["monthly"], cfg["indices"]):
            continue
        pct, _, _ = monthly_group_percent_difference(cfg, exp)
        all_vals.append(pct)
        mean_diff = np.nanmean(pct)
        status = "DA" if cfg["group"] in ASSIMILATED_GROUPS_BY_EXPERIMENT[exp] else "MO"
        ax.plot(
            months,
            pct,
            color=GROUP_COLORS[cfg["group"]],
            linestyle=GROUP_LINESTYLES[cfg["group"]],
            linewidth=2.5 if status == "DA" else 2.0,
            label=f"{status}: {cfg['group']} ({mean_diff:.2f})",
        )

    finite = np.concatenate([v[np.isfinite(v)] for v in all_vals])
    ymin = min(-5.0, np.nanpercentile(finite, 2) - 2.0)
    ymax = max(2.0, np.nanpercentile(finite, 98) + 2.0)
    ax.set_ylim(ymin, ymax)
    ax.axhline(0, color="black", linestyle=":", linewidth=1.0)
    ax.set_ylabel("Difference (%)")
    ax.set_title(f"Normalized difference of StdDev of O-F residuals (({RUN_LABELS[exp]} - OL) / OL)", loc="center")
    format_time_axis(ax)
    ax.legend(frameon=True, facecolor="white", edgecolor="gray", framealpha=0.85, loc="best")

add_panel_labels(axes)
axes[-1].set_xlabel("Month")
fig.suptitle("Monthly O-F stddev relative difference by experiment", y=1.02, fontsize=13)
savefig(fig, "fig02_experiment_monthly_omf_stdv_relative_difference")


## Fig. 3: experiment-first full-period O-F stddev relative-difference maps

Full-period `(experiment - OL) / OL * 100` by tile. Blue/negative means lower O-F stddev than OL; red/positive means higher O-F stddev; white marks near-zero differences.


In [ ]:
map_values = []
for exp in PRIMARY_EXPERIMENTS:
    for cfg in EVALUATIONS:
        vals, _, _ = temporal_group_percent_difference(cfg, exp)
        map_values.append(vals)

finite = np.concatenate([v[np.isfinite(v)] for v in map_values])
lim = np.nanpercentile(np.abs(finite), 97.5)
lim = float(np.clip(lim, 15, 30))
cmap_obj, norm = segmented_cmap_norm("RdBu_r", lim)

fig = plt.figure(figsize=(14, 6.8), constrained_layout=True)
gs = fig.add_gridspec(len(PRIMARY_EXPERIMENTS), len(EVALUATIONS))
last_sc = None
map_axes = []

for i, exp in enumerate(PRIMARY_EXPERIMENTS):
    for j, cfg in enumerate(EVALUATIONS):
        ax = make_map_axis(fig, gs[i, j])
        map_axes.append(ax)
        vals, _, _ = temporal_group_percent_difference(cfg, exp)
        status = "DA" if cfg["group"] in ASSIMILATED_GROUPS_BY_EXPERIMENT[exp] else "MO"
        mean_val = area_weighted_mean(vals)
        std_val = area_weighted_std(vals)
        last_sc = scatter_map(ax, lon, lat, vals, cmap=cmap_obj, norm=norm, s=1.05)
        ax.set_title(
            f"({RUN_LABELS[exp]} - OL) / OL, O-F StdDev {cfg['group']} ({status})\n"
            f"Mean: {mean_val:.2f} +/- {std_val:.2f} %",
            fontsize=10,
        )
add_panel_labels(map_axes)

if last_sc is not None:
    cbar = fig.colorbar(last_sc, ax=map_axes, orientation="horizontal", shrink=0.56, pad=0.05)
    cbar.set_label("%")
    set_diverging_colorbar_ticks(cbar, lim)

fig.suptitle("Full-period O-F stddev relative difference by experiment", y=1.03, fontsize=13)
savefig(fig, "fig03_experiment_full_period_omf_stdv_relative_difference_maps")


## Fig. 4: experiment-first monthly O-F stddev values

Absolute monthly O-F stddev for OL, DA H121, and DA legacy. Lower values are better; ASCAT species groups use the left axis (`m3 m-3`) and SMAP Tb uses the right axis (`K`).


In [ ]:
ABSOLUTE_STDDEV_EXPERIMENTS = (
    ("OL", "OL references"),
    ("DA_H121", RUN_LABELS["DA_H121"]),
    ("DA_legacy", RUN_LABELS["DA_legacy"]),
)


def monthly_omf_stdv_for_experiment(cfg, exp_key):
    run_name = cfg["baseline"] if exp_key == "OL" else exp_key
    return monthly_group_metric(cfg, run_name, metric="OmF_stdv")


left_values = []
right_values = []
for exp_key, _ in ABSOLUTE_STDDEV_EXPERIMENTS:
    for cfg in EVALUATIONS:
        vals = monthly_omf_stdv_for_experiment(cfg, exp_key)
        if cfg["group"] == "SMAP":
            right_values.append(vals)
        else:
            left_values.append(vals)

left_finite = np.concatenate([v[np.isfinite(v)] for v in left_values])
right_finite = np.concatenate([v[np.isfinite(v)] for v in right_values])
left_ylim = (0.0, float(np.nanpercentile(left_finite, 99) * 1.12))
right_ylim = (0.0, float(np.nanpercentile(right_finite, 99) * 1.12))

fig, axes = plt.subplots(len(ABSOLUTE_STDDEV_EXPERIMENTS), 1, figsize=(13, 9.4), sharex=True, constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, (exp_key, exp_label) in zip(axes, ABSOLUTE_STDDEV_EXPERIMENTS):
    ax_smap = ax.twinx()
    handles = []
    labels = []

    for cfg in EVALUATIONS:
        vals = monthly_omf_stdv_for_experiment(cfg, exp_key)
        target_ax = ax_smap if cfg["group"] == "SMAP" else ax
        if exp_key == "OL":
            status = "OL"
        else:
            status = "DA" if cfg["group"] in ASSIMILATED_GROUPS_BY_EXPERIMENT[exp_key] else "MO"
        mean_val = np.nanmean(vals)
        unit_label = "K" if cfg["group"] == "SMAP" else "m3 m-3"
        line = target_ax.plot(
            months,
            vals,
            color=GROUP_COLORS[cfg["group"]],
            linestyle=GROUP_LINESTYLES[cfg["group"]],
            linewidth=2.6 if status == "DA" else 2.1,
            label=f"{status}: {cfg['group']} ({mean_val:.3g} {unit_label})",
        )[0]
        handles.append(line)
        labels.append(line.get_label())

    ax.set_ylim(*left_ylim)
    ax_smap.set_ylim(*right_ylim)
    ax.set_ylabel("ASCAT O-F stddev\n(m3 m-3)")
    ax_smap.set_ylabel("SMAP O-F stddev\n(K)")
    ax.set_title(f"Monthly O-F stddev: {exp_label}", loc="center")
    format_time_axis(ax)
    ax_smap.grid(False)
    ax.legend(handles, labels, frameon=True, facecolor="white", edgecolor="gray", framealpha=0.85, loc="best")

add_panel_labels(axes)
axes[-1].set_xlabel("Month")
fig.suptitle("Monthly O-F stddev by experiment", y=1.02, fontsize=13)
savefig(fig, "fig04_experiment_monthly_omf_stdv_values_dual_axis")


## Fig. 4b: experiment-first monthly O-F mean values

Absolute monthly O-F mean for OL, DA H121, and DA legacy. Values near zero indicate smaller mean O-F bias; positive means observations are wetter/higher than forecasts, and negative means observations are drier/lower than forecasts. ASCAT species groups use the left axis (`m3 m-3`) and SMAP Tb uses the right axis (`K`).


In [ ]:
def monthly_omf_mean_for_experiment(cfg, exp_key):
    run_name = cfg["baseline"] if exp_key == "OL" else exp_key
    return monthly_group_metric(cfg, run_name, metric="OmF_mean")


left_values = []
right_values = []
for exp_key, _ in ABSOLUTE_STDDEV_EXPERIMENTS:
    for cfg in EVALUATIONS:
        vals = monthly_omf_mean_for_experiment(cfg, exp_key)
        if cfg["group"] == "SMAP":
            right_values.append(vals)
        else:
            left_values.append(vals)

left_finite = np.concatenate([v[np.isfinite(v)] for v in left_values])
right_finite = np.concatenate([v[np.isfinite(v)] for v in right_values])
left_abs = float(np.nanpercentile(np.abs(left_finite), 99) * 1.12)
right_abs = float(np.nanpercentile(np.abs(right_finite), 99) * 1.12)
left_ylim = (-left_abs, left_abs)
right_ylim = (-right_abs, right_abs)

fig, axes = plt.subplots(len(ABSOLUTE_STDDEV_EXPERIMENTS), 1, figsize=(13, 9.4), sharex=True, constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, (exp_key, exp_label) in zip(axes, ABSOLUTE_STDDEV_EXPERIMENTS):
    ax_smap = ax.twinx()
    handles = []
    labels = []

    for cfg in EVALUATIONS:
        vals = monthly_omf_mean_for_experiment(cfg, exp_key)
        target_ax = ax_smap if cfg["group"] == "SMAP" else ax
        if exp_key == "OL":
            status = "OL"
        else:
            status = "DA" if cfg["group"] in ASSIMILATED_GROUPS_BY_EXPERIMENT[exp_key] else "MO"
        mean_val = np.nanmean(vals)
        unit_label = "K" if cfg["group"] == "SMAP" else "m3 m-3"
        line = target_ax.plot(
            months,
            vals,
            color=GROUP_COLORS[cfg["group"]],
            linestyle=GROUP_LINESTYLES[cfg["group"]],
            linewidth=2.6 if status == "DA" else 2.1,
            label=f"{status}: {cfg['group']} ({mean_val:.3g} {unit_label})",
        )[0]
        handles.append(line)
        labels.append(line.get_label())

    ax.axhline(0, color="0.25", linewidth=0.8)
    ax_smap.axhline(0, color="0.25", linewidth=0.8, alpha=0.0)
    ax.set_ylim(*left_ylim)
    ax_smap.set_ylim(*right_ylim)
    ax.set_ylabel("ASCAT O-F mean\n(m3 m-3)")
    ax_smap.set_ylabel("SMAP O-F mean\n(K)")
    ax.set_title(f"Monthly O-F mean: {exp_label}", loc="center")
    format_time_axis(ax)
    ax_smap.grid(False)
    ax.legend(handles, labels, frameon=True, facecolor="white", edgecolor="gray", framealpha=0.85, loc="best")

add_panel_labels(axes)
axes[-1].set_xlabel("Month")
fig.suptitle("Monthly O-F mean by experiment", y=1.02, fontsize=13)
savefig(fig, "fig04b_experiment_monthly_omf_mean_values_dual_axis")


## Support A: observation support by species group

Observation density in the matching OL/background files. Yellow/green indicates denser observation support; this is context rather than a skill direction.


In [ ]:
density = {}
for cfg in EVALUATIONS:
    data = runs[cfg["baseline"]]["temporal"]
    density[cfg["group"]] = grouped_count(data["N_data"], cfg["indices"]) / period_days

positive = np.concatenate([v[np.isfinite(v) & (v > 0)] for v in density.values()])
vmin = max(np.nanpercentile(positive, 2), 1e-3)
vmax = np.nanpercentile(positive, 98)
cmap_count, norm_count, count_bounds = segmented_log_cmap_norm("viridis", vmin, vmax)

fig = plt.figure(figsize=(7.4, 11.8), constrained_layout=True)
gs = fig.add_gridspec(len(EVALUATIONS), 1)
last_sc = None
map_axes = []
for i, cfg in enumerate(EVALUATIONS):
    ax = make_map_axis(fig, gs[i, 0])
    map_axes.append(ax)
    vals = density[cfg["group"]]
    last_sc = scatter_map(ax, lon, lat, vals, cmap=cmap_count, norm=norm_count, s=1.1)
    ax.set_title(f"{cfg['group']}\nmedian={np.nanmedian(vals):.2g} obs/day")

add_panel_labels(map_axes)

if last_sc is not None:
    cbar = fig.colorbar(last_sc, ax=map_axes, orientation="vertical", shrink=0.78, pad=0.03, boundaries=count_bounds, spacing="uniform")
    tick_idx = np.linspace(0, len(count_bounds) - 1, 4).round().astype(int)
    ticks = count_bounds[tick_idx]
    cbar.set_ticks(ticks)
    cbar.ax.set_yticklabels([f"{t:.2g}" for t in ticks])
    cbar.set_label("Observations per tile per day")

fig.suptitle("Full-period observation density by species group", y=1.01, fontsize=13)
savefig(fig, "supportA_combined_observation_density_maps")


## Support B: monthly observation counts by species group

Monthly observation counts in the matching OL/background files. Higher lines indicate more observations, not better skill.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.0), constrained_layout=True)
family_colors = {"SMAP": "#4c78a8", "legacy ASCAT": "#f58518", "H121 ASCAT": "#54a24b"}

for cfg in EVALUATIONS:
    data = runs[cfg["baseline"]]["monthly"]
    counts = grouped_count(data["N_data"], cfg["indices"])
    ax.plot(months, counts / 1e6, marker="o", ms=3.0, linewidth=1.8, color=family_colors[cfg["group"]], label=cfg["group"])

ax.set_ylabel("Monthly observations (million)")
ax.set_xlabel("Month")
ax.set_title("Monthly observation counts in matching OL/background files", loc="left")
format_time_axis(ax)
ax.legend(ncol=3, frameon=False)
add_panel_labels([ax])
savefig(fig, "supportB_combined_monthly_observation_counts")


## Support C: OL ASCAT O/F mean time series

Monthly OL monitor means for the ASCAT observation species groups. Higher/lower values are soil-moisture state differences rather than a skill direction. These monthly inputs are already aggregated by species group, so they cannot be recomputed on the H121/legacy common tile mask from this cache.


In [ ]:
ASCAT_OL_FAMILIES = [
    {
        "group": "legacy ASCAT",
        "short": "Legacy",
        "run": "OL_vs_legacyobs",
        "cfg": next(cfg for cfg in EVALUATIONS if cfg["group"] == "legacy ASCAT"),
    },
    {
        "group": "H121 ASCAT",
        "short": "H121",
        "run": "OL_vs_H121obs",
        "cfg": next(cfg for cfg in EVALUATIONS if cfg["group"] == "H121 ASCAT"),
    },
]
OL_MEAN_METRICS = [("O_mean", "O mean"), ("F_mean", "F mean")]

ol_mean_timeseries_rows = []
fig, axes = plt.subplots(len(OL_MEAN_METRICS), 1, figsize=(12.5, 6.4), sharex=True, constrained_layout=True)
axes = np.atleast_1d(axes)

for ax, (metric, metric_label) in zip(axes, OL_MEAN_METRICS):
    for fam in ASCAT_OL_FAMILIES:
        vals = monthly_group_metric(fam["cfg"], fam["run"], metric=metric)
        mean_val = np.nanmean(vals)
        ax.plot(
            months,
            vals,
            color=GROUP_COLORS[fam["group"]],
            linestyle=GROUP_LINESTYLES[fam["group"]],
            linewidth=2.2,
            label=f"{fam['short']} ({mean_val:.3f})",
        )
        for month, value in zip(months, vals):
            ol_mean_timeseries_rows.append(
                {
                    "month": month.strftime("%Y-%m"),
                    "family": fam["group"],
                    "run": fam["run"],
                    "metric": metric,
                    "metric_label": metric_label,
                    "value": value,
                }
            )
    ax.set_ylabel("Mean (m3 m-3)")
    ax.set_title(metric_label, loc="left")
    format_time_axis(ax)
    ax.legend(frameon=False, ncol=2)

axes[-1].set_xlabel("Month")
add_panel_labels(axes)
fig.suptitle("OL ASCAT observation and forecast means", y=1.02, fontsize=13)
savefig(fig, "supportC_ol_ascat_o_f_mean_timeseries")

ol_mean_timeseries = pd.DataFrame(ol_mean_timeseries_rows)
ol_mean_timeseries_path = OUT_DIR / "ol_ascat_o_f_mean_timeseries.csv"
ol_mean_timeseries.to_csv(ol_mean_timeseries_path, index=False)
ol_mean_timeseries.head()


## Support D: OL ASCAT O/F mean maps

Full-period OL monitor tile means for ASCAT O and F. Darker colors indicate wetter mean soil moisture, not better skill.


In [ ]:
ol_mean_map_values = {}
ol_mean_map_rows = []
for fam in ASCAT_OL_FAMILIES:
    for metric, metric_label in OL_MEAN_METRICS:
        vals = temporal_group_metric(fam["cfg"], fam["run"], metric=metric)
        ol_mean_map_values[(fam["group"], metric)] = vals
        ol_mean_map_rows.append(
            {
                "family": fam["group"],
                "run": fam["run"],
                "metric": metric,
                "metric_label": metric_label,
                "mean": area_weighted_mean(vals),
                "median": np.nanmedian(vals),
                "p05": np.nanpercentile(vals, 5),
                "p95": np.nanpercentile(vals, 95),
                "n_valid_tiles": spatial_valid_count(vals),
            }
        )

finite_abs = np.concatenate([vals[np.isfinite(vals)] for vals in ol_mean_map_values.values()])
abs_vmin = 0.0
abs_vmax = float(np.nanpercentile(finite_abs, 98.5))
abs_vmax = float(np.clip(abs_vmax, 0.45, 0.65))
cmap_abs, norm_abs, abs_bounds = segmented_linear_cmap_norm("YlGnBu", abs_vmin, abs_vmax, n_bins=12)

fig = plt.figure(figsize=(13.5, 7.4), constrained_layout=True)
gs = fig.add_gridspec(len(ASCAT_OL_FAMILIES), len(OL_MEAN_METRICS))
map_axes = []
last_sc = None
for r, fam in enumerate(ASCAT_OL_FAMILIES):
    for c, (metric, metric_label) in enumerate(OL_MEAN_METRICS):
        ax = make_map_axis(fig, gs[r, c])
        map_axes.append(ax)
        vals = ol_mean_map_values[(fam["group"], metric)]
        last_sc = scatter_map(ax, lon, lat, vals, cmap=cmap_abs, norm=norm_abs, s=1.05)
        ax.set_title(
            f"{fam['short']} OL {metric_label}\n"
            f"mean={area_weighted_mean(vals):.3f}; n={spatial_valid_count(vals):,}",
            fontsize=10,
        )

add_panel_labels(map_axes)
if last_sc is not None:
    cbar = fig.colorbar(last_sc, ax=map_axes, orientation="horizontal", shrink=0.62, pad=0.05, boundaries=abs_bounds, spacing="uniform")
    ticks = np.linspace(abs_vmin, abs_vmax, 5)
    cbar.set_ticks(ticks)
    cbar.ax.set_xticklabels([f"{t:.2f}" for t in ticks])
    cbar.set_label("Mean soil moisture (m3 m-3)")

fig.suptitle("OL ASCAT O and F full-period means", y=1.03, fontsize=13)
savefig(fig, "supportD_ol_ascat_o_f_mean_maps")

ol_mean_map_summary = pd.DataFrame(ol_mean_map_rows)
ol_mean_map_summary_path = OUT_DIR / "ol_ascat_o_f_mean_map_summary.csv"
ol_mean_map_summary.to_csv(ol_mean_map_summary_path, index=False)
ol_mean_map_summary


## Support D2: OL ASCAT O/F mean maps, cross-masked

Same four full-period OL monitor maps as Support D, but every panel is restricted to the common finite H121/legacy O/F tile support. This makes absolute O and F levels comparable on the same map cells used for paired H121-versus-legacy diagnostics.


In [ ]:
cross_mask_keys = [
    (fam["group"], metric)
    for fam in ASCAT_OL_FAMILIES
    for metric, _ in OL_MEAN_METRICS
]

ol_mean_cross_mask = np.ones_like(lat, dtype=bool)
for key in cross_mask_keys:
    ol_mean_cross_mask &= np.isfinite(ol_mean_map_values[key])
ol_mean_cross_mask &= np.isfinite(lat) & (lat >= MAP_LAT_MIN) & np.isfinite(tile_area) & (tile_area > 0)

# Support E's O-difference support is the paired finite O support. The full O/F cross-mask
# should match it if O and F share support in these OL monitor summaries.
support_e_o_mask = (
    np.isfinite(ol_mean_map_values[("H121 ASCAT", "O_mean")])
    & np.isfinite(ol_mean_map_values[("legacy ASCAT", "O_mean")])
    & np.isfinite(lat)
    & (lat >= MAP_LAT_MIN)
    & np.isfinite(tile_area)
    & (tile_area > 0)
)
common_n = int(ol_mean_cross_mask.sum())
support_e_o_n = int(support_e_o_mask.sum())
print(f"Common H121/legacy O/F map cells: {common_n:,}")
print(f"Support E paired O map cells: {support_e_o_n:,}")
if common_n != support_e_o_n:
    print(f"Note: O/F cross-mask removes {support_e_o_n - common_n:,} cells relative to the paired-O mask.")

ol_mean_cross_map_values = {
    key: np.where(ol_mean_cross_mask, vals, np.nan)
    for key, vals in ol_mean_map_values.items()
}

cross_finite_abs = np.concatenate([vals[np.isfinite(vals)] for vals in ol_mean_cross_map_values.values()])
cross_abs_vmin = 0.0
cross_abs_vmax = float(np.nanpercentile(cross_finite_abs, 98.5))
cross_abs_vmax = float(np.clip(cross_abs_vmax, 0.45, 0.65))
cmap_cross_abs, norm_cross_abs, cross_abs_bounds = segmented_linear_cmap_norm("YlGnBu", cross_abs_vmin, cross_abs_vmax, n_bins=12)

fig = plt.figure(figsize=(13.5, 7.4), constrained_layout=True)
gs = fig.add_gridspec(len(ASCAT_OL_FAMILIES), len(OL_MEAN_METRICS))
map_axes = []
last_sc = None
for r, fam in enumerate(ASCAT_OL_FAMILIES):
    for c, (metric, metric_label) in enumerate(OL_MEAN_METRICS):
        ax = make_map_axis(fig, gs[r, c])
        map_axes.append(ax)
        vals = ol_mean_cross_map_values[(fam["group"], metric)]
        last_sc = scatter_map(ax, lon, lat, vals, cmap=cmap_cross_abs, norm=norm_cross_abs, s=1.05)
        ax.set_title(
            f"{fam['short']} OL {metric_label}\n"
            f"mean={area_weighted_mean(vals):.3f}; n={spatial_valid_count(vals):,}",
            fontsize=10,
        )

add_panel_labels(map_axes)
if last_sc is not None:
    cbar = fig.colorbar(last_sc, ax=map_axes, orientation="horizontal", shrink=0.62, pad=0.05, boundaries=cross_abs_bounds, spacing="uniform")
    ticks = np.linspace(cross_abs_vmin, cross_abs_vmax, 5)
    cbar.set_ticks(ticks)
    cbar.ax.set_xticklabels([f"{t:.2f}" for t in ticks])
    cbar.set_label("Mean soil moisture (m3 m-3)")

fig.suptitle("OL ASCAT O and F full-period means, common H121/legacy O/F support", y=1.03, fontsize=13)
savefig(fig, "supportD2_ol_ascat_o_f_mean_maps_cross_masked")

ol_mean_cross_map_rows = []
for fam in ASCAT_OL_FAMILIES:
    for metric, metric_label in OL_MEAN_METRICS:
        vals = ol_mean_cross_map_values[(fam["group"], metric)]
        ol_mean_cross_map_rows.append(
            {
                "species_group": fam["group"],
                "run": fam["run"],
                "metric": metric,
                "metric_label": metric_label,
                "mean": area_weighted_mean(vals),
                "median": np.nanmedian(vals),
                "p05": np.nanpercentile(vals, 5),
                "p95": np.nanpercentile(vals, 95),
                "n_valid_tiles": spatial_valid_count(vals),
                "paired_o_n_valid_tiles": support_e_o_n,
            }
        )

ol_mean_cross_map_summary = pd.DataFrame(ol_mean_cross_map_rows)
ol_mean_cross_map_summary_path = OUT_DIR / "ol_ascat_o_f_mean_map_cross_masked_summary.csv"
ol_mean_cross_map_summary.to_csv(ol_mean_cross_map_summary_path, index=False)
ol_mean_cross_map_summary


## Support E: OL H121 minus legacy O mean map and latitude profile

Full-period H121 minus legacy OL observation mean. Red/positive means H121 O is wetter than legacy O; blue/negative means H121 O is drier; white marks near-zero differences. The right panel shows area-weighted 5-degree latitude-band means aligned to the map latitude coordinate.


In [ ]:
delta_o_mean = ol_mean_map_values[("H121 ASCAT", "O_mean")] - ol_mean_map_values[("legacy ASCAT", "O_mean")]
finite_delta = delta_o_mean[np.isfinite(delta_o_mean)]
delta_vlim = float(np.nanpercentile(np.abs(finite_delta), 98.0))
delta_vlim = float(np.clip(delta_vlim, 0.04, 0.08))
cmap_delta, norm_delta = segmented_cmap_norm("RdBu_r", delta_vlim, n_bins=12)

lat_band_edges = np.arange(MAP_LAT_MIN, MAP_EXTENT[3] + 0.1, 5.0)
lat_band_rows = []
for lo, hi in zip(lat_band_edges[:-1], lat_band_edges[1:]):
    in_band = (lat >= lo) & (lat < hi if hi < lat_band_edges[-1] else lat <= hi)
    ok = in_band & np.isfinite(delta_o_mean) & np.isfinite(tile_area) & (tile_area > 0)
    lat_band_rows.append(
        {
            "lat_min": float(lo),
            "lat_max": float(hi),
            "lat_center": float((lo + hi) / 2.0),
            "delta_o_mean": area_weighted_mean(delta_o_mean[ok], weights=tile_area[ok], latitudes=None) if ok.any() else np.nan,
            "n_valid_tiles": int(ok.sum()),
        }
    )

ol_delta_o_lat_bands = pd.DataFrame(lat_band_rows)
profile_ok = np.isfinite(ol_delta_o_lat_bands["delta_o_mean"].to_numpy())
profile_delta = ol_delta_o_lat_bands.loc[profile_ok, "delta_o_mean"].to_numpy(dtype=float)
profile_lat = ol_delta_o_lat_bands.loc[profile_ok, "lat_center"].to_numpy(dtype=float)


def profile_y_from_lat(latitudes):
    latitudes = np.asarray(latitudes, dtype=float)
    if ccrs is None:
        return latitudes
    pts = ccrs.Robinson().transform_points(ccrs.PlateCarree(), np.zeros_like(latitudes), latitudes)
    return pts[:, 1]


profile_y = profile_y_from_lat(profile_lat)
map_y_limits = profile_y_from_lat(np.array([MAP_EXTENT[2], MAP_EXTENT[3]], dtype=float))
lat_ticks = np.arange(MAP_EXTENT[2], MAP_EXTENT[3] + 1, 20.0)
y_ticks = profile_y_from_lat(lat_ticks)
profile_xlim = float(np.nanmax(np.abs(profile_delta)) * 1.18) if profile_delta.size else 0.01
profile_xlim = max(profile_xlim, 0.01)

fig = plt.figure(figsize=(12.0, 4.9), constrained_layout=True)
gs = fig.add_gridspec(1, 2, width_ratios=[4.7, 1.35])
ax_map = make_map_axis(fig, gs[0, 0])
ax_profile = fig.add_subplot(gs[0, 1])

sc = scatter_map(ax_map, lon, lat, delta_o_mean, cmap=cmap_delta, norm=norm_delta, s=1.1)
ax_map.set_title(
    "OL H121 - legacy O mean\n"
    f"mean={area_weighted_mean(delta_o_mean):.3f}; n={spatial_valid_count(delta_o_mean):,}",
    fontsize=10,
)

ax_profile.axvline(0, color="0.25", linewidth=0.8)
ax_profile.plot(profile_delta, profile_y, color="0.2", linewidth=1.4, zorder=2)
ax_profile.scatter(
    profile_delta,
    profile_y,
    c=np.where(profile_delta >= 0, "#c94f44", "#2f78b7"),
    edgecolors="0.2",
    linewidths=0.35,
    s=28,
    zorder=3,
)
ax_profile.fill_betweenx(profile_y, 0, profile_delta, where=profile_delta >= 0, color="#c94f44", alpha=0.22)
ax_profile.fill_betweenx(profile_y, 0, profile_delta, where=profile_delta < 0, color="#2f78b7", alpha=0.22)
ax_profile.set_ylim(float(map_y_limits[0]), float(map_y_limits[1]))
ax_profile.set_xlim(-profile_xlim, profile_xlim)
ax_profile.set_yticks(y_ticks)
ax_profile.set_yticklabels([f"{int(t)}" for t in lat_ticks])
ax_profile.set_xlabel("Delta O mean\n(m3 m-3)")
ax_profile.set_ylabel("Latitude")
ax_profile.set_title("5 deg latitude bands", loc="left", fontsize=10)
ax_profile.grid(True, color="0.88", linewidth=0.8)

add_panel_labels([ax_map, ax_profile])
cbar = fig.colorbar(sc, ax=[ax_map], orientation="horizontal", shrink=0.72, pad=0.08)
cbar.set_label("Delta O mean (m3 m-3)")
set_diverging_colorbar_ticks(cbar, delta_vlim, fmt="{:.2f}")
savefig(fig, "supportE_ol_h121_minus_legacy_o_mean_map")

ol_delta_o_summary = pd.DataFrame(
    [
        {
            "comparison": "H121 ASCAT - legacy ASCAT",
            "metric": "O_mean",
            "mean": area_weighted_mean(delta_o_mean),
            "median": np.nanmedian(delta_o_mean),
            "std": area_weighted_std(delta_o_mean),
            "p05": np.nanpercentile(delta_o_mean, 5),
            "p95": np.nanpercentile(delta_o_mean, 95),
            "n_valid_tiles": spatial_valid_count(delta_o_mean),
        }
    ]
)
ol_delta_o_summary_path = OUT_DIR / "ol_h121_minus_legacy_o_mean_delta_summary.csv"
ol_delta_o_summary.to_csv(ol_delta_o_summary_path, index=False)

ol_delta_o_lat_bands_path = OUT_DIR / "ol_h121_minus_legacy_o_mean_lat_bands.csv"
ol_delta_o_lat_bands.to_csv(ol_delta_o_lat_bands_path, index=False)
ol_delta_o_summary, ol_delta_o_lat_bands.head()


## Support E2: OL H121 minus legacy F mean map and latitude profile

Full-period H121 minus legacy OL forecast mean. Red/positive means H121 F is wetter than legacy F; blue/negative means H121 F is drier; white marks near-zero differences. The right panel shows area-weighted 5-degree latitude-band means aligned to the map latitude coordinate.


In [ ]:
delta_f_mean = ol_mean_map_values[("H121 ASCAT", "F_mean")] - ol_mean_map_values[("legacy ASCAT", "F_mean")]
finite_delta_f = delta_f_mean[np.isfinite(delta_f_mean)]
delta_f_vlim = float(np.nanpercentile(np.abs(finite_delta_f), 98.0))
delta_f_vlim = float(np.clip(delta_f_vlim, 0.04, 0.08))
cmap_delta_f, norm_delta_f = segmented_cmap_norm("RdBu_r", delta_f_vlim, n_bins=12)

lat_band_edges = np.arange(MAP_LAT_MIN, MAP_EXTENT[3] + 0.1, 5.0)
lat_band_rows = []
for lo, hi in zip(lat_band_edges[:-1], lat_band_edges[1:]):
    in_band = (lat >= lo) & (lat < hi if hi < lat_band_edges[-1] else lat <= hi)
    ok = in_band & np.isfinite(delta_f_mean) & np.isfinite(tile_area) & (tile_area > 0)
    lat_band_rows.append(
        {
            "lat_min": float(lo),
            "lat_max": float(hi),
            "lat_center": float((lo + hi) / 2.0),
            "delta_f_mean": area_weighted_mean(delta_f_mean[ok], weights=tile_area[ok], latitudes=None) if ok.any() else np.nan,
            "n_valid_tiles": int(ok.sum()),
        }
    )

ol_delta_f_lat_bands = pd.DataFrame(lat_band_rows)
profile_ok = np.isfinite(ol_delta_f_lat_bands["delta_f_mean"].to_numpy())
profile_delta_f = ol_delta_f_lat_bands.loc[profile_ok, "delta_f_mean"].to_numpy(dtype=float)
profile_lat_f = ol_delta_f_lat_bands.loc[profile_ok, "lat_center"].to_numpy(dtype=float)
profile_y_f = profile_y_from_lat(profile_lat_f)
map_y_limits = profile_y_from_lat(np.array([MAP_EXTENT[2], MAP_EXTENT[3]], dtype=float))
lat_ticks = np.arange(MAP_EXTENT[2], MAP_EXTENT[3] + 1, 20.0)
y_ticks = profile_y_from_lat(lat_ticks)
profile_xlim_f = float(np.nanmax(np.abs(profile_delta_f)) * 1.18) if profile_delta_f.size else 0.01
profile_xlim_f = max(profile_xlim_f, 0.01)

fig = plt.figure(figsize=(12.0, 4.9), constrained_layout=True)
gs = fig.add_gridspec(1, 2, width_ratios=[4.7, 1.35])
ax_map = make_map_axis(fig, gs[0, 0])
ax_profile = fig.add_subplot(gs[0, 1])

sc = scatter_map(ax_map, lon, lat, delta_f_mean, cmap=cmap_delta_f, norm=norm_delta_f, s=1.1)
ax_map.set_title(
    "OL H121 - legacy F mean\n"
    f"mean={area_weighted_mean(delta_f_mean):.3f}; n={spatial_valid_count(delta_f_mean):,}",
    fontsize=10,
)

ax_profile.axvline(0, color="0.25", linewidth=0.8)
ax_profile.plot(profile_delta_f, profile_y_f, color="0.2", linewidth=1.4, zorder=2)
ax_profile.scatter(
    profile_delta_f,
    profile_y_f,
    c=np.where(profile_delta_f >= 0, "#c94f44", "#2f78b7"),
    edgecolors="0.2",
    linewidths=0.35,
    s=28,
    zorder=3,
)
ax_profile.fill_betweenx(profile_y_f, 0, profile_delta_f, where=profile_delta_f >= 0, color="#c94f44", alpha=0.22)
ax_profile.fill_betweenx(profile_y_f, 0, profile_delta_f, where=profile_delta_f < 0, color="#2f78b7", alpha=0.22)
ax_profile.set_ylim(float(map_y_limits[0]), float(map_y_limits[1]))
ax_profile.set_xlim(-profile_xlim_f, profile_xlim_f)
ax_profile.set_yticks(y_ticks)
ax_profile.set_yticklabels([f"{int(t)}" for t in lat_ticks])
ax_profile.set_xlabel("Delta F mean\n(m3 m-3)")
ax_profile.set_ylabel("Latitude")
ax_profile.set_title("5 deg latitude bands", loc="left", fontsize=10)
ax_profile.grid(True, color="0.88", linewidth=0.8)

add_panel_labels([ax_map, ax_profile])
cbar = fig.colorbar(sc, ax=[ax_map], orientation="horizontal", shrink=0.72, pad=0.08)
cbar.set_label("Delta F mean (m3 m-3)")
set_diverging_colorbar_ticks(cbar, delta_f_vlim, fmt="{:.2f}")
savefig(fig, "supportE2_ol_h121_minus_legacy_f_mean_map")

ol_delta_f_summary = pd.DataFrame(
    [
        {
            "comparison": "H121 ASCAT - legacy ASCAT",
            "metric": "F_mean",
            "mean": area_weighted_mean(delta_f_mean),
            "median": np.nanmedian(delta_f_mean),
            "std": area_weighted_std(delta_f_mean),
            "p05": np.nanpercentile(delta_f_mean, 5),
            "p95": np.nanpercentile(delta_f_mean, 95),
            "n_valid_tiles": spatial_valid_count(delta_f_mean),
        }
    ]
)
ol_delta_f_summary_path = OUT_DIR / "ol_h121_minus_legacy_f_mean_delta_summary.csv"
ol_delta_f_summary.to_csv(ol_delta_f_summary_path, index=False)

ol_delta_f_lat_bands_path = OUT_DIR / "ol_h121_minus_legacy_f_mean_lat_bands.csv"
ol_delta_f_lat_bands.to_csv(ol_delta_f_lat_bands_path, index=False)
ol_delta_f_summary, ol_delta_f_lat_bands.head()


## Support F: seasonal OL H121 minus legacy O mean

Seasonal H121 minus legacy OL `O_mean` from the monthly OL summaries. Red/positive means H121 O is wetter than legacy O; blue/negative means H121 O is drier. Only complete three-month seasons are shown; DJF is labeled by the January/February year.


In [ ]:
SEASON_MONTHS = {
    "DJF": (12, 1, 2),
    "MAM": (3, 4, 5),
    "JJA": (6, 7, 8),
    "SON": (9, 10, 11),
}
SEASON_ORDER = list(SEASON_MONTHS)


def month_to_season(ts):
    month = int(ts.month)
    year = int(ts.year)
    if month in (12, 1, 2):
        return "DJF", year + 1 if month == 12 else year
    if month in (3, 4, 5):
        return "MAM", year
    if month in (6, 7, 8):
        return "JJA", year
    return "SON", year


seasonal_family_rows = []
for fam in ASCAT_OL_FAMILIES:
    data = runs[fam["run"]]["monthly"]
    values = monthly_group_metric(fam["cfg"], fam["run"], metric="O_mean")
    counts = grouped_count(data["N_data"], fam["cfg"]["indices"])
    family_monthly = pd.DataFrame({"month": months, "value": values, "weight": counts})
    family_monthly[["season", "season_year"]] = family_monthly["month"].apply(lambda t: pd.Series(month_to_season(t)))
    family_monthly["calendar_month"] = family_monthly["month"].dt.month

    for (season, season_year), sub in family_monthly.groupby(["season", "season_year"], sort=False):
        expected = set(SEASON_MONTHS[season])
        observed = set(sub.loc[np.isfinite(sub["value"]) & np.isfinite(sub["weight"]), "calendar_month"])
        if observed != expected:
            continue
        ok = np.isfinite(sub["value"]) & np.isfinite(sub["weight"]) & (sub["weight"] > 0)
        if not ok.any():
            continue
        seasonal_family_rows.append(
            {
                "family": fam["group"],
                "run": fam["run"],
                "season": season,
                "season_year": int(season_year),
                "o_mean": float(np.sum(sub.loc[ok, "value"] * sub.loc[ok, "weight"]) / np.sum(sub.loc[ok, "weight"])),
                "n_obs": float(np.sum(sub.loc[ok, "weight"])),
                "n_months": int(ok.sum()),
            }
        )

seasonal_family = pd.DataFrame(seasonal_family_rows)
legacy = seasonal_family[seasonal_family["family"] == "legacy ASCAT"].rename(columns={"o_mean": "legacy_o_mean", "n_obs": "legacy_n_obs"})
h121 = seasonal_family[seasonal_family["family"] == "H121 ASCAT"].rename(columns={"o_mean": "h121_o_mean", "n_obs": "h121_n_obs"})
ol_seasonal_o_delta = legacy[["season", "season_year", "legacy_o_mean", "legacy_n_obs"]].merge(
    h121[["season", "season_year", "h121_o_mean", "h121_n_obs"]],
    on=["season", "season_year"],
    how="inner",
)
ol_seasonal_o_delta["delta_o_mean"] = ol_seasonal_o_delta["h121_o_mean"] - ol_seasonal_o_delta["legacy_o_mean"]
ol_seasonal_o_delta["season"] = pd.Categorical(ol_seasonal_o_delta["season"], categories=SEASON_ORDER, ordered=True)
ol_seasonal_o_delta = ol_seasonal_o_delta.sort_values(["season", "season_year"]).reset_index(drop=True)

fig, axes = plt.subplots(2, 2, figsize=(12.5, 7.2), sharey=True, constrained_layout=True)
axes = axes.ravel()
for ax, season in zip(axes, SEASON_ORDER):
    sub = ol_seasonal_o_delta[ol_seasonal_o_delta["season"] == season]
    colors = np.where(sub["delta_o_mean"].to_numpy() >= 0, "#c94f44", "#2f78b7")
    ax.bar(sub["season_year"], sub["delta_o_mean"], color=colors, edgecolor="0.25", linewidth=0.5, width=0.72)
    ax.axhline(0, color="0.2", linewidth=0.8)
    ax.set_title(f"{season}: H121 - legacy O mean", loc="left")
    ax.set_xlabel("Season year")
    ax.grid(True, axis="y", color="0.88", linewidth=0.8)
    if not sub.empty:
        ax.set_xticks(sub["season_year"].astype(int).to_numpy())
        ax.tick_params(axis="x", rotation=45)

for ax in axes[::2]:
    ax.set_ylabel("Delta O mean (m3 m-3)")

add_panel_labels(axes)
fig.suptitle("Seasonal OL H121-minus-legacy observation mean", y=1.03, fontsize=13)
savefig(fig, "supportF_ol_seasonal_h121_minus_legacy_o_mean")

ol_seasonal_o_delta_path = OUT_DIR / "ol_seasonal_h121_minus_legacy_o_mean.csv"
ol_seasonal_o_delta.to_csv(ol_seasonal_o_delta_path, index=False)
ol_seasonal_o_delta


## Support G: OL H121 versus legacy O mean scatter

Tile-level full-period OL `O_mean` comparison. Points above the 1:1 line have wetter H121 O than legacy O; darker bins contain more land area.


In [ ]:
legacy_o = ol_mean_map_values[("legacy ASCAT", "O_mean")]
h121_o = ol_mean_map_values[("H121 ASCAT", "O_mean")]
valid_scatter = (
    np.isfinite(legacy_o)
    & np.isfinite(h121_o)
    & np.isfinite(tile_area)
    & (tile_area > 0)
    & np.isfinite(lat)
    & (lat >= MAP_LAT_MIN)
)
scatter_weights = tile_area[valid_scatter]
scatter_total_area = float(np.sum(scatter_weights))
scatter_values = np.concatenate([legacy_o[valid_scatter], h121_o[valid_scatter]])
xy_max = float(np.nanpercentile(scatter_values, 99.5))
xy_max = float(np.clip(xy_max, 0.45, 0.65))
xy_min = 0.0
xy_bins = np.linspace(xy_min, xy_max, 76)
area_hist, xedges, yedges = np.histogram2d(
    legacy_o[valid_scatter],
    h121_o[valid_scatter],
    bins=[xy_bins, xy_bins],
    weights=scatter_weights,
)
area_hist_pct = np.divide(100.0 * area_hist, scatter_total_area, out=np.zeros_like(area_hist), where=scatter_total_area > 0)
positive_area = area_hist_pct[area_hist_pct > 0]
scatter_vmin = max(float(np.nanpercentile(positive_area, 5)), 1e-4)
scatter_vmax = float(np.nanpercentile(positive_area, 99.2))
cmap_scatter, norm_scatter, scatter_bounds = segmented_log_cmap_norm("magma", scatter_vmin, scatter_vmax, n_bins=10)

fig, ax = plt.subplots(figsize=(6.8, 6.0), constrained_layout=True)
mesh = ax.pcolormesh(
    xedges,
    yedges,
    np.ma.masked_where(area_hist_pct.T <= 0, area_hist_pct.T),
    cmap=cmap_scatter,
    norm=norm_scatter,
)
ax.plot([xy_min, xy_max], [xy_min, xy_max], color="0.2", linewidth=1.0, linestyle="--")
mean_delta = area_weighted_mean(h121_o - legacy_o)
median_delta = np.nanmedian(h121_o[valid_scatter] - legacy_o[valid_scatter])
ax.set_xlim(xy_min, xy_max)
ax.set_ylim(xy_min, xy_max)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("Legacy OL O mean (m3 m-3)")
ax.set_ylabel("H121 OL O mean (m3 m-3)")
ax.set_title(f"Tile O mean comparison\nmean delta={mean_delta:.3f}; median delta={median_delta:.3f}", fontsize=10)
ax.grid(True, color="0.9", linewidth=0.7)
add_panel_labels([ax])
cbar = fig.colorbar(mesh, ax=ax, shrink=0.82, boundaries=scatter_bounds, spacing="uniform")
cbar.set_label("Land area in bin (%)")
savefig(fig, "supportG_ol_h121_vs_legacy_o_mean_scatter")

ol_o_mean_scatter_summary = pd.DataFrame(
    [
        {
            "metric": "O_mean",
            "mean_delta": mean_delta,
            "median_delta": median_delta,
            "n_valid_tiles": int(valid_scatter.sum()),
        }
    ]
)
ol_o_mean_scatter_summary_path = OUT_DIR / "ol_h121_vs_legacy_o_mean_scatter_summary.csv"
ol_o_mean_scatter_summary.to_csv(ol_o_mean_scatter_summary_path, index=False)
ol_o_mean_scatter_summary


## Support H: OL H121 minus legacy O mean distribution

Distribution of full-period `H121 - legacy` OL `O_mean` across land area. Red/positive means H121 O is wetter than legacy O; blue/negative means H121 O is drier.


In [ ]:
valid_delta_o = (
    np.isfinite(delta_o_mean)
    & np.isfinite(tile_area)
    & (tile_area > 0)
    & np.isfinite(lat)
    & (lat >= MAP_LAT_MIN)
)
delta_o_valid = delta_o_mean[valid_delta_o]
delta_o_weights = tile_area[valid_delta_o]
hist_vlim = float(np.nanpercentile(np.abs(delta_o_valid), 99.0))
hist_vlim = float(np.clip(hist_vlim, 0.06, 0.09))
hist_bins = np.linspace(-hist_vlim, hist_vlim, 41)
clipped_delta = np.clip(delta_o_valid, hist_bins[0] + np.finfo(float).eps, hist_bins[-1] - np.finfo(float).eps)
hist_area, _ = np.histogram(clipped_delta, bins=hist_bins, weights=delta_o_weights)
hist_pct = 100.0 * hist_area / np.sum(delta_o_weights)
hist_centers = 0.5 * (hist_bins[:-1] + hist_bins[1:])
hist_width = np.diff(hist_bins)

order = np.argsort(delta_o_valid)
cdf_x = delta_o_valid[order]
cdf_y = 100.0 * np.cumsum(delta_o_weights[order]) / np.sum(delta_o_weights)
mean_delta = area_weighted_mean(delta_o_mean)
median_delta = np.interp(50.0, cdf_y, cdf_x)

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.6), constrained_layout=True)
bar_colors = np.where(hist_centers >= 0, "#c94f44", "#2f78b7")
axes[0].bar(hist_centers, hist_pct, width=hist_width, align="center", color=bar_colors, edgecolor="0.25", linewidth=0.35)
axes[0].axvline(0, color="0.2", linewidth=0.8)
axes[0].axvline(mean_delta, color="0.1", linestyle="--", linewidth=1.0, label=f"mean {mean_delta:.3f}")
axes[0].set_xlim(-hist_vlim, hist_vlim)
axes[0].set_xlabel("Delta O mean (m3 m-3)")
axes[0].set_ylabel("Land area (%)")
axes[0].set_title("Histogram", loc="left")
axes[0].grid(True, axis="y", color="0.88", linewidth=0.8)
axes[0].legend(frameon=False)

axes[1].plot(cdf_x, cdf_y, color="0.2", linewidth=1.8)
axes[1].axvline(0, color="0.2", linewidth=0.8)
axes[1].axvline(median_delta, color="0.1", linestyle="--", linewidth=1.0, label=f"median {median_delta:.3f}")
axes[1].set_xlim(-hist_vlim, hist_vlim)
axes[1].set_ylim(0, 100)
axes[1].set_xlabel("Delta O mean (m3 m-3)")
axes[1].set_ylabel("Cumulative land area (%)")
axes[1].set_title("CDF", loc="left")
axes[1].grid(True, color="0.88", linewidth=0.8)
axes[1].legend(frameon=False, loc="lower right")
add_panel_labels(axes)
fig.suptitle("OL H121-minus-legacy observation mean distribution", y=1.03, fontsize=13)
savefig(fig, "supportH_ol_h121_minus_legacy_o_mean_distribution")

ol_o_mean_histogram = pd.DataFrame(
    {
        "bin_min": hist_bins[:-1],
        "bin_max": hist_bins[1:],
        "bin_center": hist_centers,
        "land_area_percent": hist_pct,
    }
)
ol_o_mean_histogram_path = OUT_DIR / "ol_h121_minus_legacy_o_mean_histogram.csv"
ol_o_mean_histogram.to_csv(ol_o_mean_histogram_path, index=False)

ol_o_mean_cdf = pd.DataFrame({"delta_o_mean": cdf_x, "cumulative_land_area_percent": cdf_y})
ol_o_mean_cdf_path = OUT_DIR / "ol_h121_minus_legacy_o_mean_cdf.csv"
ol_o_mean_cdf.to_csv(ol_o_mean_cdf_path, index=False)
ol_o_mean_histogram.head(), ol_o_mean_cdf.head()


## Support I: OL H121 minus legacy O-F bias

Full-period delta in OL `O-F` mean: `(O-F)_H121 - (O-F)_legacy`. Red/positive means the H121 O-F mean is higher than legacy; blue/negative means it is lower; white marks near-zero differences.


In [ ]:
legacy_omf_mean = temporal_group_metric(ASCAT_OL_FAMILIES[0]["cfg"], ASCAT_OL_FAMILIES[0]["run"], metric="OmF_mean")
h121_omf_mean = temporal_group_metric(ASCAT_OL_FAMILIES[1]["cfg"], ASCAT_OL_FAMILIES[1]["run"], metric="OmF_mean")
delta_omf_mean = h121_omf_mean - legacy_omf_mean
finite_bias = delta_omf_mean[np.isfinite(delta_omf_mean)]
bias_vlim = float(np.nanpercentile(np.abs(finite_bias), 98.0))
bias_vlim = float(np.clip(bias_vlim, 0.01, 0.025))
cmap_bias, norm_bias = segmented_cmap_norm("RdBu_r", bias_vlim, n_bins=12)

bias_lat_bands = latitude_band_summary(delta_omf_mean, band_width=5.0).rename(columns={"value": "delta_omf_mean"})
profile_ok = np.isfinite(bias_lat_bands["delta_omf_mean"].to_numpy())
profile_delta = bias_lat_bands.loc[profile_ok, "delta_omf_mean"].to_numpy(dtype=float)
profile_lat = bias_lat_bands.loc[profile_ok, "lat_center"].to_numpy(dtype=float)
profile_y = profile_y_from_lat(profile_lat)
map_y_limits = profile_y_from_lat(np.array([MAP_EXTENT[2], MAP_EXTENT[3]], dtype=float))
lat_ticks = np.arange(MAP_EXTENT[2], MAP_EXTENT[3] + 1, 20.0)
y_ticks = profile_y_from_lat(lat_ticks)
profile_xlim = float(np.nanmax(np.abs(profile_delta)) * 1.18) if profile_delta.size else 0.01
profile_xlim = max(profile_xlim, 0.006)

fig = plt.figure(figsize=(12.0, 4.9), constrained_layout=True)
gs = fig.add_gridspec(1, 2, width_ratios=[4.7, 1.35])
ax_map = make_map_axis(fig, gs[0, 0])
ax_profile = fig.add_subplot(gs[0, 1])

sc = scatter_map(ax_map, lon, lat, delta_omf_mean, cmap=cmap_bias, norm=norm_bias, s=1.1)
ax_map.set_title(
    "OL H121 - legacy O-F mean\n"
    f"mean={area_weighted_mean(delta_omf_mean):.4f}; n={spatial_valid_count(delta_omf_mean):,}",
    fontsize=10,
)

ax_profile.axvline(0, color="0.25", linewidth=0.8)
ax_profile.plot(profile_delta, profile_y, color="0.2", linewidth=1.4, zorder=2)
ax_profile.scatter(
    profile_delta,
    profile_y,
    c=np.where(profile_delta >= 0, "#c94f44", "#2f78b7"),
    edgecolors="0.2",
    linewidths=0.35,
    s=28,
    zorder=3,
)
ax_profile.fill_betweenx(profile_y, 0, profile_delta, where=profile_delta >= 0, color="#c94f44", alpha=0.22)
ax_profile.fill_betweenx(profile_y, 0, profile_delta, where=profile_delta < 0, color="#2f78b7", alpha=0.22)
ax_profile.set_ylim(float(map_y_limits[0]), float(map_y_limits[1]))
ax_profile.set_xlim(-profile_xlim, profile_xlim)
ax_profile.set_yticks(y_ticks)
ax_profile.set_yticklabels([f"{int(t)}" for t in lat_ticks])
ax_profile.set_xlabel("Delta O-F mean\n(m3 m-3)")
ax_profile.set_ylabel("Latitude")
ax_profile.set_title("5 deg latitude bands", loc="left", fontsize=10)
ax_profile.grid(True, color="0.88", linewidth=0.8)

add_panel_labels([ax_map, ax_profile])
cbar = fig.colorbar(sc, ax=[ax_map], orientation="horizontal", shrink=0.72, pad=0.08)
cbar.set_label("Delta O-F mean (m3 m-3)")
set_diverging_colorbar_ticks(cbar, bias_vlim, fmt="{:.3f}")
savefig(fig, "supportI_ol_h121_minus_legacy_omf_mean_map")

ol_delta_omf_summary = pd.DataFrame(
    [
        {
            "comparison": "H121 ASCAT - legacy ASCAT",
            "metric": "OmF_mean",
            "mean": area_weighted_mean(delta_omf_mean),
            "median": np.nanmedian(delta_omf_mean),
            "std": area_weighted_std(delta_omf_mean),
            "p05": np.nanpercentile(delta_omf_mean, 5),
            "p95": np.nanpercentile(delta_omf_mean, 95),
            "n_valid_tiles": spatial_valid_count(delta_omf_mean),
        }
    ]
)
ol_delta_omf_summary_path = OUT_DIR / "ol_h121_minus_legacy_omf_mean_delta_summary.csv"
ol_delta_omf_summary.to_csv(ol_delta_omf_summary_path, index=False)

ol_delta_omf_lat_bands_path = OUT_DIR / "ol_h121_minus_legacy_omf_mean_lat_bands.csv"
bias_lat_bands.to_csv(ol_delta_omf_lat_bands_path, index=False)
ol_delta_omf_summary, bias_lat_bands.head()


## Support J: OL H121 minus legacy O mean by Metop platform

Full-period H121 minus legacy OL `O_mean` for each matched Metop platform. Red/positive means H121 O is wetter than the corresponding legacy platform; blue/negative means it is drier.


In [ ]:
PLATFORM_PAIRS = [
    {"platform": "Metop A", "legacy_index": 4, "h121_index": 7},
    {"platform": "Metop B", "legacy_index": 5, "h121_index": 8},
    {"platform": "Metop C", "legacy_index": 6, "h121_index": 9},
]
legacy_temporal = runs["OL_vs_legacyobs"]["temporal"]
h121_temporal = runs["OL_vs_H121obs"]["temporal"]
platform_delta_values = {}
platform_rows = []
for pair in PLATFORM_PAIRS:
    legacy_vals = species_value(legacy_temporal["O_mean"], legacy_temporal["N_data"], pair["legacy_index"])
    h121_vals = species_value(h121_temporal["O_mean"], h121_temporal["N_data"], pair["h121_index"])
    delta_vals = h121_vals - legacy_vals
    platform_delta_values[pair["platform"]] = delta_vals
    platform_rows.append(
        {
            "platform": pair["platform"],
            "legacy_species": SPECIES_10[pair["legacy_index"]],
            "h121_species": SPECIES_10[pair["h121_index"]],
            "metric": "O_mean",
            "mean": area_weighted_mean(delta_vals),
            "median": np.nanmedian(delta_vals),
            "std": area_weighted_std(delta_vals),
            "p05": np.nanpercentile(delta_vals, 5),
            "p95": np.nanpercentile(delta_vals, 95),
            "n_valid_tiles": spatial_valid_count(delta_vals),
        }
    )

finite_platform = np.concatenate([vals[np.isfinite(vals)] for vals in platform_delta_values.values()])
platform_vlim = float(np.nanpercentile(np.abs(finite_platform), 98.0))
platform_vlim = float(np.clip(platform_vlim, 0.04, 0.08))
cmap_platform, norm_platform = segmented_cmap_norm("RdBu_r", platform_vlim, n_bins=12)

fig = plt.figure(figsize=(13.5, 4.7), constrained_layout=True)
gs = fig.add_gridspec(1, len(PLATFORM_PAIRS))
map_axes = []
last_sc = None
for col, pair in enumerate(PLATFORM_PAIRS):
    ax = make_map_axis(fig, gs[0, col])
    map_axes.append(ax)
    vals = platform_delta_values[pair["platform"]]
    last_sc = scatter_map(ax, lon, lat, vals, cmap=cmap_platform, norm=norm_platform, s=1.05)
    ax.set_title(
        f"{pair['platform']}: H121 - legacy O mean\n"
        f"mean={area_weighted_mean(vals):.3f}; n={spatial_valid_count(vals):,}",
        fontsize=10,
    )

add_panel_labels(map_axes)
if last_sc is not None:
    cbar = fig.colorbar(last_sc, ax=map_axes, orientation="horizontal", shrink=0.72, pad=0.07)
    cbar.set_label("Delta O mean (m3 m-3)")
    set_diverging_colorbar_ticks(cbar, platform_vlim, fmt="{:.2f}")
fig.suptitle("OL H121-minus-legacy observation mean by Metop platform", y=1.03, fontsize=13)
savefig(fig, "supportJ_ol_platform_h121_minus_legacy_o_mean_maps")

ol_platform_delta_summary = pd.DataFrame(platform_rows)
ol_platform_delta_summary_path = OUT_DIR / "ol_platform_h121_minus_legacy_o_mean_summary.csv"
ol_platform_delta_summary.to_csv(ol_platform_delta_summary_path, index=False)
ol_platform_delta_summary


## Support K: jointly matched OL legacy-vs-H121 ASCAT setup

These additive diagnostics use `OL_legacy_h121_xmask`, which samples legacy and H121 ASCAT species pairs from the same OL run and the same tile/cycle support. In this dataset, `O_mean` is raw ASCAT wetness, while `F_mean` is the OL model forecast in soil-moisture units. Red means H121 is larger than legacy; blue means H121 is smaller. With true joint matching, H121-minus-legacy `F_mean` should be close to zero while raw `O_mean` captures the observation-product difference.


In [ ]:
XMASK_RUN = "OL_legacy_h121_xmask"
if XMASK_RUN not in runs:
    runs[XMASK_RUN] = load_run(XMASK_RUN)

XMASK_PLATFORM_PAIRS = [
    {"platform": "Metop A", "legacy_index": 4, "h121_index": 7},
    {"platform": "Metop B", "legacy_index": 5, "h121_index": 8},
    {"platform": "Metop C", "legacy_index": 6, "h121_index": 9},
]
XMASK_ASCAT_FAMILIES = [
    {
        "group": "legacy ASCAT",
        "short": "Legacy",
        "run": XMASK_RUN,
        "cfg": next(cfg for cfg in EVALUATIONS if cfg["group"] == "legacy ASCAT"),
    },
    {
        "group": "H121 ASCAT",
        "short": "H121",
        "run": XMASK_RUN,
        "cfg": next(cfg for cfg in EVALUATIONS if cfg["group"] == "H121 ASCAT"),
    },
]
XMASK_OL_MEAN_METRICS = [("O_mean", "O mean"), ("F_mean", "F mean")]
XMASK_MEAN_UNITS = {"O_mean": "raw ASCAT wetness units", "F_mean": "m3 m-3"}
XMASK_MEAN_YLABELS = {"O_mean": "Mean raw ASCAT wetness", "F_mean": "Mean model soil moisture (m3 m-3)"}
XMASK_DELTA_LABELS = {"O_mean": "Delta raw ASCAT wetness", "F_mean": "Delta model soil moisture (m3 m-3)", "OmF_mean": "Delta mixed O-F (wetness - m3 m-3)"}

xmask_temporal = runs[XMASK_RUN]["temporal"]
xmask_monthly = runs[XMASK_RUN]["monthly"]
xmask_months = xmask_monthly["months"]
if "profile_y_from_lat" not in globals():
    def profile_y_from_lat(latitudes):
        latitudes = np.asarray(latitudes, dtype=float)
        if ccrs is None:
            return latitudes
        pts = ccrs.Robinson().transform_points(ccrs.PlateCarree(), np.zeros_like(latitudes), latitudes)
        return pts[:, 1]

xmask_pair_rows = []
for pair in XMASK_PLATFORM_PAIRS:
    legacy_index = pair["legacy_index"]
    h121_index = pair["h121_index"]
    temporal_n_diff = xmask_temporal["N_data"][:, legacy_index] - xmask_temporal["N_data"][:, h121_index]
    monthly_n_diff = xmask_monthly["N_data"][:, legacy_index] - xmask_monthly["N_data"][:, h121_index]
    xmask_pair_rows.append(
        {
            "platform": pair["platform"],
            "legacy_species": SPECIES_10[legacy_index],
            "h121_species": SPECIES_10[h121_index],
            "temporal_max_abs_N_diff": np.nanmax(np.abs(temporal_n_diff)),
            "temporal_nonzero_N_diff": int(np.count_nonzero(np.nan_to_num(temporal_n_diff) != 0)),
            "monthly_max_abs_N_diff": np.nanmax(np.abs(monthly_n_diff)),
            "monthly_nonzero_N_diff": int(np.count_nonzero(np.nan_to_num(monthly_n_diff) != 0)),
        }
    )

xmask_pair_summary = pd.DataFrame(xmask_pair_rows)
xmask_pair_summary_path = OUT_DIR / "xmask_ol_legacy_h121_pair_summary.csv"
xmask_pair_summary.to_csv(xmask_pair_summary_path, index=False)
display(xmask_pair_summary)


## Support K-A: jointly matched observation support by ASCAT species group

Observation density in `OL_legacy_h121_xmask`. Yellow/green indicates denser observation support; this is context rather than a skill direction. Legacy and H121 support should match by construction because each paired species contributes only when both are present on the same tile and cycle.


In [ ]:
xmask_density = {}
xmask_period_days = days_covered(xmask_months)
for fam in XMASK_ASCAT_FAMILIES:
    xmask_density[fam["group"]] = grouped_count(xmask_temporal["N_data"], fam["cfg"]["indices"]) / xmask_period_days

positive = np.concatenate([v[np.isfinite(v) & (v > 0)] for v in xmask_density.values()])
xmask_count_vmin = max(np.nanpercentile(positive, 2), 1e-3)
xmask_count_vmax = np.nanpercentile(positive, 98)
cmap_count, norm_count, count_bounds = segmented_log_cmap_norm("viridis", xmask_count_vmin, xmask_count_vmax)

fig = plt.figure(figsize=(7.4, 8.2), constrained_layout=True)
gs = fig.add_gridspec(len(XMASK_ASCAT_FAMILIES), 1)
last_sc = None
map_axes = []
xmask_density_rows = []
for i, fam in enumerate(XMASK_ASCAT_FAMILIES):
    ax = make_map_axis(fig, gs[i, 0])
    map_axes.append(ax)
    vals = xmask_density[fam["group"]]
    last_sc = scatter_map(ax, lon, lat, vals, cmap=cmap_count, norm=norm_count, s=1.1)
    ax.set_title(f"{fam['group']} jointly matched\nmedian={np.nanmedian(vals):.2g} obs/day")
    xmask_density_rows.append(
        {
            "family": fam["group"],
            "run": XMASK_RUN,
            "mean_obs_per_tile_per_day": area_weighted_mean(vals),
            "median_obs_per_tile_per_day": np.nanmedian(vals),
            "p05_obs_per_tile_per_day": np.nanpercentile(vals, 5),
            "p95_obs_per_tile_per_day": np.nanpercentile(vals, 95),
            "n_valid_tiles": spatial_valid_count(vals),
        }
    )

add_panel_labels(map_axes)

if last_sc is not None:
    cbar = fig.colorbar(last_sc, ax=map_axes, orientation="vertical", shrink=0.78, pad=0.03, boundaries=count_bounds, spacing="uniform")
    tick_idx = np.linspace(0, len(count_bounds) - 1, 4).round().astype(int)
    ticks = count_bounds[tick_idx]
    cbar.set_ticks(ticks)
    cbar.ax.set_yticklabels([f"{t:.2g}" for t in ticks])
    cbar.set_label("Observations per tile per day")

fig.suptitle("Jointly matched full-period ASCAT observation density", y=1.01, fontsize=13)
savefig(fig, "supportK_A_xmask_observation_density_maps")

xmask_density_summary = pd.DataFrame(xmask_density_rows)
xmask_density_summary_path = OUT_DIR / "xmask_observation_density_summary.csv"
xmask_density_summary.to_csv(xmask_density_summary_path, index=False)
xmask_density_summary


## Support K-B: monthly observation counts with jointly matched ASCAT

Monthly observation counts from the original matching OL/background files, plus the jointly matched ASCAT support. Higher lines indicate more observations, not better skill.


In [ ]:
xmask_count_rows = []
fig, ax = plt.subplots(figsize=(11, 4.0), constrained_layout=True)
family_colors = {"SMAP": "#4c78a8", "legacy ASCAT": "#f58518", "H121 ASCAT": "#54a24b"}

for cfg in EVALUATIONS:
    data = runs[cfg["baseline"]]["monthly"]
    counts = grouped_count(data["N_data"], cfg["indices"])
    ax.plot(months, counts / 1e6, marker="o", ms=3.0, linewidth=1.8, color=family_colors[cfg["group"]], label=cfg["group"])
    for month, value in zip(months, counts):
        xmask_count_rows.append(
            {
                "month": month.strftime("%Y-%m"),
                "support": cfg["group"],
                "run": cfg["baseline"],
                "count": value,
            }
        )

legacy_xmask_counts = grouped_count(xmask_monthly["N_data"], XMASK_ASCAT_FAMILIES[0]["cfg"]["indices"])
h121_xmask_counts = grouped_count(xmask_monthly["N_data"], XMASK_ASCAT_FAMILIES[1]["cfg"]["indices"])
xmask_counts = legacy_xmask_counts
if not np.allclose(legacy_xmask_counts, h121_xmask_counts, equal_nan=True):
    print("Warning: xmasked legacy and H121 grouped counts differ; plotting their mean.")
    xmask_counts = np.nanmean(np.vstack([legacy_xmask_counts, h121_xmask_counts]), axis=0)

ax.plot(
    xmask_months,
    xmask_counts / 1e6,
    marker="s",
    ms=3.2,
    linewidth=2.2,
    color="0.15",
    linestyle="--",
    label="jointly matched ASCAT",
)
for month, value in zip(xmask_months, xmask_counts):
    xmask_count_rows.append(
        {
            "month": month.strftime("%Y-%m"),
            "support": "jointly matched ASCAT",
            "run": XMASK_RUN,
            "count": value,
        }
    )

ax.set_ylabel("Monthly observations (million)")
ax.set_xlabel("Month")
ax.set_title("Monthly observation counts, including jointly matched ASCAT", loc="left")
format_time_axis(ax)
ax.legend(ncol=4, frameon=False)
add_panel_labels([ax])
savefig(fig, "supportK_B_xmask_monthly_observation_counts")

xmask_count_timeseries = pd.DataFrame(xmask_count_rows)
xmask_count_timeseries_path = OUT_DIR / "xmask_monthly_observation_counts.csv"
xmask_count_timeseries.to_csv(xmask_count_timeseries_path, index=False)
xmask_count_timeseries.head()


## Support K-C: jointly matched OL ASCAT O/F mean time series

Monthly OL monitor means for the jointly matched ASCAT species groups. `O_mean` is raw ASCAT wetness; `F_mean` is model soil moisture (`m3 m-3`). Higher/lower values are state differences rather than a skill direction.


In [ ]:
xmask_ol_mean_timeseries_rows = []
fig, axes = plt.subplots(len(XMASK_OL_MEAN_METRICS), 1, figsize=(12.5, 6.4), sharex=True, constrained_layout=True)
axes = np.atleast_1d(axes)

for ax, (metric, metric_label) in zip(axes, XMASK_OL_MEAN_METRICS):
    for fam in XMASK_ASCAT_FAMILIES:
        vals = monthly_group_metric(fam["cfg"], fam["run"], metric=metric)
        mean_val = np.nanmean(vals)
        ax.plot(
            xmask_months,
            vals,
            color=GROUP_COLORS[fam["group"]],
            linestyle=GROUP_LINESTYLES[fam["group"]],
            linewidth=2.2,
            label=f"{fam['short']} ({mean_val:.3f})",
        )
        for month, value in zip(xmask_months, vals):
            xmask_ol_mean_timeseries_rows.append(
                {
                    "month": month.strftime("%Y-%m"),
                    "family": fam["group"],
                    "run": fam["run"],
                    "metric": metric,
                    "metric_label": metric_label,
                    "units": XMASK_MEAN_UNITS[metric],
                    "value": value,
                }
            )
    ax.set_ylabel(XMASK_MEAN_YLABELS[metric])
    ax.set_title(f"{metric_label} ({XMASK_MEAN_UNITS[metric]})", loc="left")
    format_time_axis(ax)
    ax.legend(frameon=False, ncol=2)

axes[-1].set_xlabel("Month")
add_panel_labels(axes)
fig.suptitle("Jointly matched OL ASCAT observation and forecast means", y=1.02, fontsize=13)
savefig(fig, "supportK_C_xmask_ol_ascat_o_f_mean_timeseries")

xmask_ol_mean_timeseries = pd.DataFrame(xmask_ol_mean_timeseries_rows)
xmask_ol_mean_timeseries_path = OUT_DIR / "xmask_ol_ascat_o_f_mean_timeseries.csv"
xmask_ol_mean_timeseries.to_csv(xmask_ol_mean_timeseries_path, index=False)
xmask_ol_mean_timeseries.head()


## Support K-C2: independent versus jointly matched OL ASCAT O/F mean time series

Monthly OL monitor means from the original independently sampled legacy/H121 OL products and the jointly matched ASCAT support. For `O_mean`, the original products are in monitored/model soil-moisture space on the left axis, while the jointly matched dataset is raw ASCAT wetness on the right axis. For `F_mean`, all lines are model soil moisture (`m3 m-3`).


In [ ]:
combined_ol_mean_timeseries_rows = []
fig, axes = plt.subplots(len(XMASK_OL_MEAN_METRICS), 1, figsize=(12.5, 6.6), sharex=True, constrained_layout=True)
axes = np.atleast_1d(axes)

combined_families = [
    {**ASCAT_OL_FAMILIES[0], "support": "independent", "linestyle": "-", "linewidth": 2.0},
    {**ASCAT_OL_FAMILIES[1], "support": "independent", "linestyle": "-", "linewidth": 2.0},
    {**XMASK_ASCAT_FAMILIES[0], "support": "jointly matched", "linestyle": "--", "linewidth": 2.2},
    {**XMASK_ASCAT_FAMILIES[1], "support": "jointly matched", "linestyle": "--", "linewidth": 2.2},
]

for ax, (metric, metric_label) in zip(axes, XMASK_OL_MEAN_METRICS):
    ax_right = ax.twinx() if metric == "O_mean" else None
    handles = []
    labels = []
    for fam in combined_families:
        vals = monthly_group_metric(fam["cfg"], fam["run"], metric=metric)
        mean_val = np.nanmean(vals)
        months_for_run = runs[fam["run"]]["monthly"]["months"]
        if metric == "O_mean" and fam["support"] == "jointly matched":
            target_ax = ax_right
            units = "raw ASCAT wetness units"
        elif metric == "O_mean":
            target_ax = ax
            units = "m3 m-3"
        else:
            target_ax = ax
            units = "m3 m-3"
        label = f"{fam['short']} {fam['support']} ({mean_val:.3f} {units})"
        line = target_ax.plot(
            months_for_run,
            vals,
            color=GROUP_COLORS[fam["group"]],
            linestyle=fam["linestyle"],
            linewidth=fam["linewidth"],
            label=label,
        )[0]
        handles.append(line)
        labels.append(label)
        for month, value in zip(months_for_run, vals):
            combined_ol_mean_timeseries_rows.append(
                {
                    "month": month.strftime("%Y-%m"),
                    "family": fam["group"],
                    "support": fam["support"],
                    "run": fam["run"],
                    "metric": metric,
                    "metric_label": metric_label,
                    "units": units,
                    "value": value,
                }
            )
    if metric == "O_mean":
        ax.set_ylabel("Independent monitored O mean\n(m3 m-3)")
        ax_right.set_ylabel("Jointly matched raw O mean\n(wetness units)")
        ax_right.grid(False)
        ax.set_title("O mean: independent monitored space vs jointly matched raw wetness", loc="left")
    else:
        ax.set_ylabel("F mean (m3 m-3)")
        ax.set_title("F mean: model soil moisture", loc="left")
    format_time_axis(ax)
    ax.legend(handles, labels, frameon=False, ncol=2)

axes[-1].set_xlabel("Month")
add_panel_labels(axes)
fig.suptitle("OL ASCAT O/F means: independent versus jointly matched support", y=1.02, fontsize=13)
savefig(fig, "supportK_C2_independent_vs_xmask_ol_ascat_o_f_mean_timeseries")

combined_ol_mean_timeseries = pd.DataFrame(combined_ol_mean_timeseries_rows)
combined_ol_mean_timeseries_path = OUT_DIR / "xmask_independent_vs_joint_ol_ascat_o_f_mean_timeseries.csv"
combined_ol_mean_timeseries.to_csv(combined_ol_mean_timeseries_path, index=False)
combined_ol_mean_timeseries.head()


## Support K-D: jointly matched OL ASCAT O/F mean maps

Full-period jointly matched OL monitor tile means for ASCAT O and F. `O_mean` is raw ASCAT wetness; `F_mean` is model soil moisture (`m3 m-3`). Darker colors indicate larger values within each column, not better skill.


In [ ]:
xmask_ol_mean_map_values = {}
xmask_ol_mean_map_rows = []
for fam in XMASK_ASCAT_FAMILIES:
    for metric, metric_label in XMASK_OL_MEAN_METRICS:
        vals = temporal_group_metric(fam["cfg"], fam["run"], metric=metric)
        xmask_ol_mean_map_values[(fam["group"], metric)] = vals
        xmask_ol_mean_map_rows.append(
            {
                "family": fam["group"],
                "run": fam["run"],
                "metric": metric,
                "metric_label": metric_label,
                "units": XMASK_MEAN_UNITS[metric],
                "mean": area_weighted_mean(vals),
                "median": np.nanmedian(vals),
                "p05": np.nanpercentile(vals, 5),
                "p95": np.nanpercentile(vals, 95),
                "n_valid_tiles": spatial_valid_count(vals),
            }
        )

xmask_abs_norms = {}
for metric, _ in XMASK_OL_MEAN_METRICS:
    vals_for_metric = [xmask_ol_mean_map_values[(fam["group"], metric)] for fam in XMASK_ASCAT_FAMILIES]
    finite_abs = np.concatenate([vals[np.isfinite(vals)] for vals in vals_for_metric])
    abs_vmin = 0.0
    abs_vmax = float(np.nanpercentile(finite_abs, 98.5))
    abs_vmax = float(np.clip(abs_vmax, 0.45, 0.65))
    xmask_abs_norms[metric] = segmented_linear_cmap_norm("YlGnBu", abs_vmin, abs_vmax, n_bins=12)

fig = plt.figure(figsize=(13.5, 7.4), constrained_layout=True)
gs = fig.add_gridspec(len(XMASK_ASCAT_FAMILIES), len(XMASK_OL_MEAN_METRICS))
map_axes = []
last_sc_by_metric = {}
for r, fam in enumerate(XMASK_ASCAT_FAMILIES):
    for c, (metric, metric_label) in enumerate(XMASK_OL_MEAN_METRICS):
        ax = make_map_axis(fig, gs[r, c])
        map_axes.append(ax)
        vals = xmask_ol_mean_map_values[(fam["group"], metric)]
        cmap_abs, norm_abs, _ = xmask_abs_norms[metric]
        last_sc_by_metric[metric] = scatter_map(ax, lon, lat, vals, cmap=cmap_abs, norm=norm_abs, s=1.05)
        ax.set_title(
            f"{fam['short']} jointly matched OL {metric_label}\n"
            f"mean={area_weighted_mean(vals):.3f}; n={spatial_valid_count(vals):,}",
            fontsize=10,
        )

add_panel_labels(map_axes)
for c, (metric, metric_label) in enumerate(XMASK_OL_MEAN_METRICS):
    sc = last_sc_by_metric.get(metric)
    if sc is None:
        continue
    _, _, bounds = xmask_abs_norms[metric]
    col_axes = [fig.axes[r * len(XMASK_OL_MEAN_METRICS) + c] for r in range(len(XMASK_ASCAT_FAMILIES))]
    cbar = fig.colorbar(sc, ax=col_axes, orientation="horizontal", shrink=0.70, pad=0.05, boundaries=bounds, spacing="uniform")
    ticks = np.linspace(bounds[0], bounds[-1], 5)
    cbar.set_ticks(ticks)
    cbar.ax.set_xticklabels([f"{t:.2f}" for t in ticks])
    cbar.set_label(f"{metric_label} ({XMASK_MEAN_UNITS[metric]})")

fig.suptitle("Jointly matched OL ASCAT O and F full-period means", y=1.03, fontsize=13)
savefig(fig, "supportK_D_xmask_ol_ascat_o_f_mean_maps")

xmask_ol_mean_map_summary = pd.DataFrame(xmask_ol_mean_map_rows)
xmask_ol_mean_map_summary_path = OUT_DIR / "xmask_ol_ascat_o_f_mean_map_summary.csv"
xmask_ol_mean_map_summary.to_csv(xmask_ol_mean_map_summary_path, index=False)
xmask_ol_mean_map_summary


## Support K-E: jointly matched OL H121 minus legacy O mean map and latitude profile

Full-period H121 minus legacy raw ASCAT wetness on the jointly matched sample. Red/positive means H121 O has higher wetness than legacy O; blue/negative means H121 O has lower wetness; white marks near-zero differences. The right panel shows area-weighted 5-degree latitude-band means aligned to the map latitude coordinate.


In [ ]:
xmask_delta_o_mean = xmask_ol_mean_map_values[("H121 ASCAT", "O_mean")] - xmask_ol_mean_map_values[("legacy ASCAT", "O_mean")]
finite_delta = xmask_delta_o_mean[np.isfinite(xmask_delta_o_mean)]
xmask_delta_vlim = float(np.nanpercentile(np.abs(finite_delta), 98.0))
xmask_delta_vlim = float(np.clip(xmask_delta_vlim, 0.04, 0.22))
cmap_delta, norm_delta = segmented_cmap_norm("RdBu_r", xmask_delta_vlim, n_bins=12)

lat_band_edges = np.arange(MAP_LAT_MIN, MAP_EXTENT[3] + 0.1, 5.0)
lat_band_rows = []
for lo, hi in zip(lat_band_edges[:-1], lat_band_edges[1:]):
    in_band = (lat >= lo) & (lat < hi if hi < lat_band_edges[-1] else lat <= hi)
    ok = in_band & np.isfinite(xmask_delta_o_mean) & np.isfinite(tile_area) & (tile_area > 0)
    lat_band_rows.append(
        {
            "lat_min": float(lo),
            "lat_max": float(hi),
            "lat_center": float((lo + hi) / 2.0),
            "delta_o_mean": area_weighted_mean(xmask_delta_o_mean[ok], weights=tile_area[ok], latitudes=None) if ok.any() else np.nan,
            "n_valid_tiles": int(ok.sum()),
        }
    )

xmask_ol_delta_o_lat_bands = pd.DataFrame(lat_band_rows)
profile_ok = np.isfinite(xmask_ol_delta_o_lat_bands["delta_o_mean"].to_numpy())
profile_delta = xmask_ol_delta_o_lat_bands.loc[profile_ok, "delta_o_mean"].to_numpy(dtype=float)
profile_lat = xmask_ol_delta_o_lat_bands.loc[profile_ok, "lat_center"].to_numpy(dtype=float)
profile_y = profile_y_from_lat(profile_lat)
map_y_limits = profile_y_from_lat(np.array([MAP_EXTENT[2], MAP_EXTENT[3]], dtype=float))
lat_ticks = np.arange(MAP_EXTENT[2], MAP_EXTENT[3] + 1, 20.0)
y_ticks = profile_y_from_lat(lat_ticks)
profile_xlim = float(np.nanmax(np.abs(profile_delta)) * 1.18) if profile_delta.size else 0.01
profile_xlim = max(profile_xlim, 0.01)

fig = plt.figure(figsize=(12.0, 4.9), constrained_layout=True)
gs = fig.add_gridspec(1, 2, width_ratios=[4.7, 1.35])
ax_map = make_map_axis(fig, gs[0, 0])
ax_profile = fig.add_subplot(gs[0, 1])

sc = scatter_map(ax_map, lon, lat, xmask_delta_o_mean, cmap=cmap_delta, norm=norm_delta, s=1.1)
ax_map.set_title(
    "Jointly matched OL H121 - legacy raw O mean\n"
    f"mean={area_weighted_mean(xmask_delta_o_mean):.3f}; n={spatial_valid_count(xmask_delta_o_mean):,}",
    fontsize=10,
)

ax_profile.axvline(0, color="0.25", linewidth=0.8)
ax_profile.plot(profile_delta, profile_y, color="0.2", linewidth=1.4, zorder=2)
ax_profile.scatter(
    profile_delta,
    profile_y,
    c=np.where(profile_delta >= 0, "#c94f44", "#2f78b7"),
    edgecolors="0.2",
    linewidths=0.35,
    s=28,
    zorder=3,
)
ax_profile.fill_betweenx(profile_y, 0, profile_delta, where=profile_delta >= 0, color="#c94f44", alpha=0.22)
ax_profile.fill_betweenx(profile_y, 0, profile_delta, where=profile_delta < 0, color="#2f78b7", alpha=0.22)
ax_profile.set_ylim(float(map_y_limits[0]), float(map_y_limits[1]))
ax_profile.set_xlim(-profile_xlim, profile_xlim)
ax_profile.set_yticks(y_ticks)
ax_profile.set_yticklabels([f"{int(t)}" for t in lat_ticks])
ax_profile.set_xlabel("Delta O mean\n(wetness units)")
ax_profile.set_ylabel("Latitude")
ax_profile.set_title("5 deg latitude bands", loc="left", fontsize=10)
ax_profile.grid(True, color="0.88", linewidth=0.8)

add_panel_labels([ax_map, ax_profile])
cbar = fig.colorbar(sc, ax=[ax_map], orientation="horizontal", shrink=0.72, pad=0.08)
cbar.set_label("Delta O mean (raw ASCAT wetness units)")
set_diverging_colorbar_ticks(cbar, xmask_delta_vlim, fmt="{:.2f}")
savefig(fig, "supportK_E_xmask_ol_h121_minus_legacy_o_mean_map")

xmask_ol_delta_o_summary = pd.DataFrame(
    [
        {
            "comparison": "H121 ASCAT - legacy ASCAT",
            "run": XMASK_RUN,
            "metric": "O_mean",
            "units": "raw ASCAT wetness units",
            "mean": area_weighted_mean(xmask_delta_o_mean),
            "median": np.nanmedian(xmask_delta_o_mean),
            "std": area_weighted_std(xmask_delta_o_mean),
            "p05": np.nanpercentile(xmask_delta_o_mean, 5),
            "p95": np.nanpercentile(xmask_delta_o_mean, 95),
            "n_valid_tiles": spatial_valid_count(xmask_delta_o_mean),
        }
    ]
)
xmask_ol_delta_o_summary_path = OUT_DIR / "xmask_ol_h121_minus_legacy_o_mean_delta_summary.csv"
xmask_ol_delta_o_summary.to_csv(xmask_ol_delta_o_summary_path, index=False)

xmask_ol_delta_o_lat_bands_path = OUT_DIR / "xmask_ol_h121_minus_legacy_o_mean_lat_bands.csv"
xmask_ol_delta_o_lat_bands.to_csv(xmask_ol_delta_o_lat_bands_path, index=False)
xmask_ol_delta_o_summary, xmask_ol_delta_o_lat_bands.head()


## Support K-E2: jointly matched OL H121 minus legacy F mean map and latitude profile

Full-period H121 minus legacy OL forecast mean on the jointly matched sample. `F_mean` is model soil moisture (`m3 m-3`). Red/positive means H121 F is wetter than legacy F; blue/negative means H121 F is drier; white marks near-zero differences. Under true joint matching this should be close to zero because both species groups sample the same OL forecast trajectory on the same support.


In [ ]:
xmask_delta_f_mean = xmask_ol_mean_map_values[("H121 ASCAT", "F_mean")] - xmask_ol_mean_map_values[("legacy ASCAT", "F_mean")]
finite_delta_f = xmask_delta_f_mean[np.isfinite(xmask_delta_f_mean)]
xmask_delta_f_vlim = float(np.nanpercentile(np.abs(finite_delta_f), 98.0))
xmask_delta_f_vlim = float(np.clip(xmask_delta_f_vlim, 0.002, 0.020))
cmap_delta_f, norm_delta_f = segmented_cmap_norm("RdBu_r", xmask_delta_f_vlim, n_bins=12)

lat_band_edges = np.arange(MAP_LAT_MIN, MAP_EXTENT[3] + 0.1, 5.0)
lat_band_rows = []
for lo, hi in zip(lat_band_edges[:-1], lat_band_edges[1:]):
    in_band = (lat >= lo) & (lat < hi if hi < lat_band_edges[-1] else lat <= hi)
    ok = in_band & np.isfinite(xmask_delta_f_mean) & np.isfinite(tile_area) & (tile_area > 0)
    lat_band_rows.append(
        {
            "lat_min": float(lo),
            "lat_max": float(hi),
            "lat_center": float((lo + hi) / 2.0),
            "delta_f_mean": area_weighted_mean(xmask_delta_f_mean[ok], weights=tile_area[ok], latitudes=None) if ok.any() else np.nan,
            "n_valid_tiles": int(ok.sum()),
        }
    )

xmask_ol_delta_f_lat_bands = pd.DataFrame(lat_band_rows)
profile_ok = np.isfinite(xmask_ol_delta_f_lat_bands["delta_f_mean"].to_numpy())
profile_delta_f = xmask_ol_delta_f_lat_bands.loc[profile_ok, "delta_f_mean"].to_numpy(dtype=float)
profile_lat_f = xmask_ol_delta_f_lat_bands.loc[profile_ok, "lat_center"].to_numpy(dtype=float)
profile_y_f = profile_y_from_lat(profile_lat_f)
map_y_limits = profile_y_from_lat(np.array([MAP_EXTENT[2], MAP_EXTENT[3]], dtype=float))
lat_ticks = np.arange(MAP_EXTENT[2], MAP_EXTENT[3] + 1, 20.0)
y_ticks = profile_y_from_lat(lat_ticks)
profile_xlim_f = float(np.nanmax(np.abs(profile_delta_f)) * 1.18) if profile_delta_f.size else 0.01
profile_xlim_f = max(profile_xlim_f, 0.002)

fig = plt.figure(figsize=(12.0, 4.9), constrained_layout=True)
gs = fig.add_gridspec(1, 2, width_ratios=[4.7, 1.35])
ax_map = make_map_axis(fig, gs[0, 0])
ax_profile = fig.add_subplot(gs[0, 1])

sc = scatter_map(ax_map, lon, lat, xmask_delta_f_mean, cmap=cmap_delta_f, norm=norm_delta_f, s=1.1)
ax_map.set_title(
    "Jointly matched OL H121 - legacy F mean\n"
    f"mean={area_weighted_mean(xmask_delta_f_mean):.4f}; n={spatial_valid_count(xmask_delta_f_mean):,}",
    fontsize=10,
)

ax_profile.axvline(0, color="0.25", linewidth=0.8)
ax_profile.plot(profile_delta_f, profile_y_f, color="0.2", linewidth=1.4, zorder=2)
ax_profile.scatter(
    profile_delta_f,
    profile_y_f,
    c=np.where(profile_delta_f >= 0, "#c94f44", "#2f78b7"),
    edgecolors="0.2",
    linewidths=0.35,
    s=28,
    zorder=3,
)
ax_profile.fill_betweenx(profile_y_f, 0, profile_delta_f, where=profile_delta_f >= 0, color="#c94f44", alpha=0.22)
ax_profile.fill_betweenx(profile_y_f, 0, profile_delta_f, where=profile_delta_f < 0, color="#2f78b7", alpha=0.22)
ax_profile.set_ylim(float(map_y_limits[0]), float(map_y_limits[1]))
ax_profile.set_xlim(-profile_xlim_f, profile_xlim_f)
ax_profile.set_yticks(y_ticks)
ax_profile.set_yticklabels([f"{int(t)}" for t in lat_ticks])
ax_profile.set_xlabel("Delta F mean\n(m3 m-3)")
ax_profile.set_ylabel("Latitude")
ax_profile.set_title("5 deg latitude bands", loc="left", fontsize=10)
ax_profile.grid(True, color="0.88", linewidth=0.8)

add_panel_labels([ax_map, ax_profile])
cbar = fig.colorbar(sc, ax=[ax_map], orientation="horizontal", shrink=0.72, pad=0.08)
cbar.set_label("Delta F mean (m3 m-3)")
set_diverging_colorbar_ticks(cbar, xmask_delta_f_vlim, fmt="{:.3f}")
savefig(fig, "supportK_E2_xmask_ol_h121_minus_legacy_f_mean_map")

xmask_ol_delta_f_summary = pd.DataFrame(
    [
        {
            "comparison": "H121 ASCAT - legacy ASCAT",
            "run": XMASK_RUN,
            "metric": "F_mean",
            "units": "m3 m-3",
            "mean": area_weighted_mean(xmask_delta_f_mean),
            "median": np.nanmedian(xmask_delta_f_mean),
            "std": area_weighted_std(xmask_delta_f_mean),
            "p05": np.nanpercentile(xmask_delta_f_mean, 5),
            "p95": np.nanpercentile(xmask_delta_f_mean, 95),
            "n_valid_tiles": spatial_valid_count(xmask_delta_f_mean),
        }
    ]
)
xmask_ol_delta_f_summary_path = OUT_DIR / "xmask_ol_h121_minus_legacy_f_mean_delta_summary.csv"
xmask_ol_delta_f_summary.to_csv(xmask_ol_delta_f_summary_path, index=False)

xmask_ol_delta_f_lat_bands_path = OUT_DIR / "xmask_ol_h121_minus_legacy_f_mean_lat_bands.csv"
xmask_ol_delta_f_lat_bands.to_csv(xmask_ol_delta_f_lat_bands_path, index=False)
xmask_ol_delta_f_summary, xmask_ol_delta_f_lat_bands.head()


## Support K-G: jointly matched OL H121 versus legacy O mean scatter

Tile-level full-period raw ASCAT wetness `O_mean` comparison on the jointly matched sample. Points above the 1:1 line have wetter H121 O than legacy O; darker bins contain more land area.


In [ ]:
xmask_legacy_o = xmask_ol_mean_map_values[("legacy ASCAT", "O_mean")]
xmask_h121_o = xmask_ol_mean_map_values[("H121 ASCAT", "O_mean")]
xmask_valid_scatter = (
    np.isfinite(xmask_legacy_o)
    & np.isfinite(xmask_h121_o)
    & np.isfinite(tile_area)
    & (tile_area > 0)
    & np.isfinite(lat)
    & (lat >= MAP_LAT_MIN)
)
xmask_scatter_weights = tile_area[xmask_valid_scatter]
xmask_scatter_total_area = float(np.sum(xmask_scatter_weights))
xmask_scatter_values = np.concatenate([xmask_legacy_o[xmask_valid_scatter], xmask_h121_o[xmask_valid_scatter]])
xy_max = float(np.nanpercentile(xmask_scatter_values, 99.5))
xy_max = float(np.clip(xy_max, 0.45, 0.65))
xy_min = 0.0
xy_bins = np.linspace(xy_min, xy_max, 76)
area_hist, xedges, yedges = np.histogram2d(
    xmask_legacy_o[xmask_valid_scatter],
    xmask_h121_o[xmask_valid_scatter],
    bins=[xy_bins, xy_bins],
    weights=xmask_scatter_weights,
)
area_hist_pct = np.divide(100.0 * area_hist, xmask_scatter_total_area, out=np.zeros_like(area_hist), where=xmask_scatter_total_area > 0)
positive_area = area_hist_pct[area_hist_pct > 0]
scatter_vmin = max(float(np.nanpercentile(positive_area, 5)), 1e-4)
scatter_vmax = float(np.nanpercentile(positive_area, 99.2))
cmap_scatter, norm_scatter, scatter_bounds = segmented_log_cmap_norm("magma", scatter_vmin, scatter_vmax, n_bins=10)

fig, ax = plt.subplots(figsize=(6.8, 6.0), constrained_layout=True)
mesh = ax.pcolormesh(
    xedges,
    yedges,
    np.ma.masked_where(area_hist_pct.T <= 0, area_hist_pct.T),
    cmap=cmap_scatter,
    norm=norm_scatter,
)
ax.plot([xy_min, xy_max], [xy_min, xy_max], color="0.2", linewidth=1.0, linestyle="--")
mean_delta = area_weighted_mean(xmask_h121_o - xmask_legacy_o)
median_delta = np.nanmedian(xmask_h121_o[xmask_valid_scatter] - xmask_legacy_o[xmask_valid_scatter])
ax.set_xlim(xy_min, xy_max)
ax.set_ylim(xy_min, xy_max)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("Legacy jointly matched raw O mean (wetness units)")
ax.set_ylabel("H121 jointly matched raw O mean (wetness units)")
ax.set_title(f"Jointly matched raw O mean comparison\nmean delta={mean_delta:.3f}; median delta={median_delta:.3f}", fontsize=10)
ax.grid(True, color="0.9", linewidth=0.7)
add_panel_labels([ax])
cbar = fig.colorbar(mesh, ax=ax, shrink=0.82, boundaries=scatter_bounds, spacing="uniform")
cbar.set_label("Land area in bin (%)")
savefig(fig, "supportK_G_xmask_ol_h121_vs_legacy_o_mean_scatter")

xmask_ol_o_mean_scatter_summary = pd.DataFrame(
    [
        {
            "run": XMASK_RUN,
            "metric": "O_mean",
            "mean_delta": mean_delta,
            "median_delta": median_delta,
            "n_valid_tiles": int(xmask_valid_scatter.sum()),
        }
    ]
)
xmask_ol_o_mean_scatter_summary_path = OUT_DIR / "xmask_ol_h121_vs_legacy_o_mean_scatter_summary.csv"
xmask_ol_o_mean_scatter_summary.to_csv(xmask_ol_o_mean_scatter_summary_path, index=False)
xmask_ol_o_mean_scatter_summary


## Support K-I: jointly matched OL H121 minus legacy mixed-unit O-F diagnostic

Full-period delta in jointly matched `O-F` mean, where `O` is raw ASCAT wetness and `F` is model soil moisture. This is a mixed-unit diagnostic, not a physical O-F bias. Red/positive means the H121 mixed O-F mean is higher than legacy; blue/negative means it is lower; white marks near-zero differences.


In [ ]:
xmask_legacy_omf_mean = temporal_group_metric(XMASK_ASCAT_FAMILIES[0]["cfg"], XMASK_RUN, metric="OmF_mean")
xmask_h121_omf_mean = temporal_group_metric(XMASK_ASCAT_FAMILIES[1]["cfg"], XMASK_RUN, metric="OmF_mean")
xmask_delta_omf_mean = xmask_h121_omf_mean - xmask_legacy_omf_mean
finite_bias = xmask_delta_omf_mean[np.isfinite(xmask_delta_omf_mean)]
xmask_bias_vlim = float(np.nanpercentile(np.abs(finite_bias), 98.0))
xmask_bias_vlim = float(np.clip(xmask_bias_vlim, 0.04, 0.22))
cmap_bias, norm_bias = segmented_cmap_norm("RdBu_r", xmask_bias_vlim, n_bins=12)

xmask_bias_lat_bands = latitude_band_summary(xmask_delta_omf_mean, band_width=5.0).rename(columns={"value": "delta_omf_mean"})
profile_ok = np.isfinite(xmask_bias_lat_bands["delta_omf_mean"].to_numpy())
profile_delta = xmask_bias_lat_bands.loc[profile_ok, "delta_omf_mean"].to_numpy(dtype=float)
profile_lat = xmask_bias_lat_bands.loc[profile_ok, "lat_center"].to_numpy(dtype=float)
profile_y = profile_y_from_lat(profile_lat)
map_y_limits = profile_y_from_lat(np.array([MAP_EXTENT[2], MAP_EXTENT[3]], dtype=float))
lat_ticks = np.arange(MAP_EXTENT[2], MAP_EXTENT[3] + 1, 20.0)
y_ticks = profile_y_from_lat(lat_ticks)
profile_xlim = float(np.nanmax(np.abs(profile_delta)) * 1.18) if profile_delta.size else 0.01
profile_xlim = max(profile_xlim, 0.01)

fig = plt.figure(figsize=(12.0, 4.9), constrained_layout=True)
gs = fig.add_gridspec(1, 2, width_ratios=[4.7, 1.35])
ax_map = make_map_axis(fig, gs[0, 0])
ax_profile = fig.add_subplot(gs[0, 1])

sc = scatter_map(ax_map, lon, lat, xmask_delta_omf_mean, cmap=cmap_bias, norm=norm_bias, s=1.1)
ax_map.set_title(
    "Jointly matched OL H121 - legacy mixed O-F mean\n"
    f"mean={area_weighted_mean(xmask_delta_omf_mean):.3f}; n={spatial_valid_count(xmask_delta_omf_mean):,}",
    fontsize=10,
)

ax_profile.axvline(0, color="0.25", linewidth=0.8)
ax_profile.plot(profile_delta, profile_y, color="0.2", linewidth=1.4, zorder=2)
ax_profile.scatter(
    profile_delta,
    profile_y,
    c=np.where(profile_delta >= 0, "#c94f44", "#2f78b7"),
    edgecolors="0.2",
    linewidths=0.35,
    s=28,
    zorder=3,
)
ax_profile.fill_betweenx(profile_y, 0, profile_delta, where=profile_delta >= 0, color="#c94f44", alpha=0.22)
ax_profile.fill_betweenx(profile_y, 0, profile_delta, where=profile_delta < 0, color="#2f78b7", alpha=0.22)
ax_profile.set_ylim(float(map_y_limits[0]), float(map_y_limits[1]))
ax_profile.set_xlim(-profile_xlim, profile_xlim)
ax_profile.set_yticks(y_ticks)
ax_profile.set_yticklabels([f"{int(t)}" for t in lat_ticks])
ax_profile.set_xlabel("Delta O-F mean\n(wetness - m3 m-3)")
ax_profile.set_ylabel("Latitude")
ax_profile.set_title("5 deg latitude bands", loc="left", fontsize=10)
ax_profile.grid(True, color="0.88", linewidth=0.8)

add_panel_labels([ax_map, ax_profile])
cbar = fig.colorbar(sc, ax=[ax_map], orientation="horizontal", shrink=0.72, pad=0.08)
cbar.set_label("Delta O-F mean (wetness - m3 m-3)")
set_diverging_colorbar_ticks(cbar, xmask_bias_vlim, fmt="{:.2f}")
savefig(fig, "supportK_I_xmask_ol_h121_minus_legacy_omf_mean_map")

xmask_ol_delta_omf_summary = pd.DataFrame(
    [
        {
            "comparison": "H121 ASCAT - legacy ASCAT",
            "run": XMASK_RUN,
            "metric": "OmF_mean",
            "units": "wetness - m3 m-3",
            "mean": area_weighted_mean(xmask_delta_omf_mean),
            "median": np.nanmedian(xmask_delta_omf_mean),
            "std": area_weighted_std(xmask_delta_omf_mean),
            "p05": np.nanpercentile(xmask_delta_omf_mean, 5),
            "p95": np.nanpercentile(xmask_delta_omf_mean, 95),
            "n_valid_tiles": spatial_valid_count(xmask_delta_omf_mean),
        }
    ]
)
xmask_ol_delta_omf_summary_path = OUT_DIR / "xmask_ol_h121_minus_legacy_omf_mean_delta_summary.csv"
xmask_ol_delta_omf_summary.to_csv(xmask_ol_delta_omf_summary_path, index=False)

xmask_ol_delta_omf_lat_bands_path = OUT_DIR / "xmask_ol_h121_minus_legacy_omf_mean_lat_bands.csv"
xmask_bias_lat_bands.to_csv(xmask_ol_delta_omf_lat_bands_path, index=False)
xmask_ol_delta_omf_summary, xmask_bias_lat_bands.head()


## Support K-J: jointly matched OL H121 minus legacy O mean by Metop platform

Full-period H121 minus legacy raw ASCAT wetness `O_mean` for each matched Metop platform on the jointly matched sample. Red/positive means H121 O has higher wetness than the corresponding legacy platform; blue/negative means it is lower.


In [ ]:
xmask_platform_delta_values = {}
xmask_platform_rows = []
for pair in XMASK_PLATFORM_PAIRS:
    legacy_vals = species_value(xmask_temporal["O_mean"], xmask_temporal["N_data"], pair["legacy_index"])
    h121_vals = species_value(xmask_temporal["O_mean"], xmask_temporal["N_data"], pair["h121_index"])
    delta_vals = h121_vals - legacy_vals
    xmask_platform_delta_values[pair["platform"]] = delta_vals
    xmask_platform_rows.append(
        {
            "platform": pair["platform"],
            "run": XMASK_RUN,
            "legacy_species": SPECIES_10[pair["legacy_index"]],
            "h121_species": SPECIES_10[pair["h121_index"]],
            "metric": "O_mean",
            "units": "raw ASCAT wetness units",
            "mean": area_weighted_mean(delta_vals),
            "median": np.nanmedian(delta_vals),
            "std": area_weighted_std(delta_vals),
            "p05": np.nanpercentile(delta_vals, 5),
            "p95": np.nanpercentile(delta_vals, 95),
            "n_valid_tiles": spatial_valid_count(delta_vals),
        }
    )

finite_platform = np.concatenate([vals[np.isfinite(vals)] for vals in xmask_platform_delta_values.values()])
xmask_platform_vlim = float(np.nanpercentile(np.abs(finite_platform), 98.0))
xmask_platform_vlim = float(np.clip(xmask_platform_vlim, 0.04, 0.22))
cmap_platform, norm_platform = segmented_cmap_norm("RdBu_r", xmask_platform_vlim, n_bins=12)

fig = plt.figure(figsize=(13.5, 4.7), constrained_layout=True)
gs = fig.add_gridspec(1, len(XMASK_PLATFORM_PAIRS))
map_axes = []
last_sc = None
for col, pair in enumerate(XMASK_PLATFORM_PAIRS):
    ax = make_map_axis(fig, gs[0, col])
    map_axes.append(ax)
    vals = xmask_platform_delta_values[pair["platform"]]
    last_sc = scatter_map(ax, lon, lat, vals, cmap=cmap_platform, norm=norm_platform, s=1.05)
    ax.set_title(
        f"{pair['platform']}: H121 - legacy raw O mean\n"
        f"mean={area_weighted_mean(vals):.3f}; n={spatial_valid_count(vals):,}",
        fontsize=10,
    )

add_panel_labels(map_axes)
if last_sc is not None:
    cbar = fig.colorbar(last_sc, ax=map_axes, orientation="horizontal", shrink=0.72, pad=0.07)
    cbar.set_label("Delta raw O mean (ASCAT wetness units)")
    set_diverging_colorbar_ticks(cbar, xmask_platform_vlim, fmt="{:.2f}")
fig.suptitle("Jointly matched OL H121-minus-legacy observation mean by Metop platform", y=1.03, fontsize=13)
savefig(fig, "supportK_J_xmask_ol_platform_h121_minus_legacy_o_mean_maps")

xmask_ol_platform_delta_summary = pd.DataFrame(xmask_platform_rows)
xmask_ol_platform_delta_summary_path = OUT_DIR / "xmask_ol_platform_h121_minus_legacy_o_mean_summary.csv"
xmask_ol_platform_delta_summary.to_csv(xmask_ol_platform_delta_summary_path, index=False)
xmask_ol_platform_delta_summary


## Species-level helper table

This table assigns each species to the baseline run that contains the corresponding observations.

In [ ]:
species_rows = []
for i, name in enumerate(SPECIES_10):
    if i < 4:
        group = "SMAP"
        baseline = "OL_vs_SMAPobs"
        experiments = ("DA_H121", "DA_legacy", "DA_SMAP")
    elif i < 7:
        group = "legacy ASCAT"
        baseline = "OL_vs_legacyobs"
        experiments = ("DA_H121", "DA_legacy", "DA_SMAP")
    else:
        group = "H121 ASCAT"
        baseline = "OL_vs_H121obs"
        experiments = ("DA_H121", "DA_legacy")
    species_rows.append({"index": i, "species": name, "group": group, "baseline": baseline, "experiments": experiments})

species_table = pd.DataFrame(species_rows)
species_table

## Fig. 5: species-level monthly O-F stddev improvement

Monthly `OL - DA` O-F stddev improvement by species. Positive values mean the DA run lowers O-F stddev; negative values mean it increases O-F stddev.


In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(13, 12), sharex=True, constrained_layout=True)
axes = axes.ravel()

for ax, row in zip(axes, species_rows):
    idx = row["index"]
    base = runs[row["baseline"]]["monthly"]
    ol = species_value(base["OmF_stdv"], base["N_data"], idx)
    for exp in species_experiment_plot_order(row):
        exp_data = runs[exp]["monthly"]
        if idx >= exp_data["OmF_stdv"].shape[-1]:
            continue
        da = species_value(exp_data["OmF_stdv"], exp_data["N_data"], idx)
        imp = percent_improvement(ol, da)
        smooth = pd.Series(imp, index=months).rolling(ROLLING_WINDOW, center=True, min_periods=1).mean()
        zorder = 5 if exp == "DA_H121" else 2
        marker = "o" if exp == "DA_H121" else None
        ax.plot(months, smooth, color=RUN_COLORS[exp], linewidth=2.0 if exp == "DA_H121" else 1.7, marker=marker, markersize=2.4 if exp == "DA_H121" else 0, markevery=6 if exp == "DA_H121" else None, zorder=zorder, label=RUN_LABELS[exp])
        ax.scatter(months, imp, s=10 if exp == "DA_H121" else 8, color=RUN_COLORS[exp], alpha=0.28 if exp == "DA_H121" else 0.18, zorder=zorder)
    ax.axhline(0, color="0.25", linewidth=0.7)
    ax.set_title(f"{row['species']} ({row['group']})", loc="left", fontsize=10)
    format_time_axis(ax)

for ax in axes[::2]:
    ax.set_ylabel("Improvement (%)")
for ax in axes[-2:]:
    ax.set_xlabel("Month")

add_panel_labels(axes)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.02))
fig.suptitle("Species-level monthly O-F stddev improvement", y=1.045, fontsize=13)
savefig(fig, "fig05_species_monthly_omf_stdv_improvement")


## Fig. 6: species-level full-period O-F stddev improvement bars

Full-period `OL - DA` O-F stddev improvement by species. Positive bars mean lower O-F stddev than OL; negative bars mean higher O-F stddev.


In [ ]:
bar_rows = []
for row in species_rows:
    idx = row["index"]
    base = runs[row["baseline"]]["temporal"]
    ol = species_value(base["OmF_stdv"], base["N_data"], idx)
    for exp in row["experiments"]:
        exp_data = runs[exp]["temporal"]
        if idx >= exp_data["OmF_stdv"].shape[-1]:
            continue
        da = species_value(exp_data["OmF_stdv"], exp_data["N_data"], idx)
        imp = percent_improvement(ol, da)
        bar_rows.append(
            {
                "species": row["species"],
                "group": row["group"],
                "experiment": exp,
                "experiment_label": RUN_LABELS[exp],
                "mean_pct_improvement": area_weighted_mean(imp),
                "median_pct_improvement": np.nanmedian(imp),
                "fraction_tiles_improved": area_weighted_fraction(imp > 0, imp),
                "n_valid_tiles": int(np.isfinite(imp).sum()),
            }
        )

species_summary = pd.DataFrame(bar_rows)
species_summary_path = OUT_DIR / "species_omf_stdv_improvement_summary.csv"
species_summary.to_csv(species_summary_path, index=False)

fig, ax = plt.subplots(figsize=(13, 5.8), constrained_layout=True)
species_order = [row["species"] for row in species_rows]
x = np.arange(len(species_order))
width = 0.24
offsets = {"DA_H121": -width, "DA_legacy": 0.0, "DA_SMAP": width}

for exp in ("DA_H121", "DA_legacy", "DA_SMAP"):
    vals = []
    for species in species_order:
        hit = species_summary[(species_summary["species"] == species) & (species_summary["experiment"] == exp)]
        vals.append(np.nan if hit.empty else float(hit["mean_pct_improvement"].iloc[0]))
    ax.bar(x + offsets[exp], vals, width=width, color=RUN_COLORS[exp], label=RUN_LABELS[exp])

ax.axhline(0, color="0.25", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(species_order, rotation=35, ha="right")
ax.set_ylabel("Mean tile improvement (%)")
ax.set_title("Full-period O-F stddev improvement by species", loc="left")
ax.grid(True, axis="y", color="0.88", linewidth=0.8)
ax.legend(ncol=3, frameon=False)
add_panel_labels([ax])
savefig(fig, "fig06_species_full_period_omf_stdv_improvement_bars")

species_summary

## All combined species-group metrics

This section writes long-form monthly values and full-period tile summaries for every metric available in the summary files, grouped into SMAP, legacy ASCAT, and H121 ASCAT species groups.

In [ ]:
MONTHLY_METRICS = [
    "O_mean",
    "O_stdv",
    "F_mean",
    "F_stdv",
    "A_mean",
    "A_stdv",
    "OmF_mean",
    "OmF_stdv",
    "OmA_mean",
    "OmA_stdv",
    "N_data",
]
TEMPORAL_METRICS = MONTHLY_METRICS + ["OmF_norm_mean", "OmF_norm_stdv"]
MEAN_METRICS = ["O_mean", "F_mean", "A_mean", "OmF_mean", "OmA_mean"]
STDV_METRICS = ["O_stdv", "F_stdv", "A_stdv", "OmF_stdv", "OmA_stdv"]

METRIC_LABELS = {
    "O_mean": "O mean",
    "O_stdv": "O stddev",
    "F_mean": "F mean",
    "F_stdv": "F stddev",
    "A_mean": "A mean",
    "A_stdv": "A stddev",
    "OmF_mean": "O-F mean",
    "OmF_stdv": "O-F stddev",
    "OmA_mean": "O-A mean",
    "OmA_stdv": "O-A stddev",
    "OmF_norm_mean": "normalized O-F mean",
    "OmF_norm_stdv": "normalized O-F stddev",
    "N_data": "observation count",
}


def grouped_metric(data, cfg, metric):
    if metric not in data or not has_species(data, cfg["indices"]):
        return None
    if metric == "N_data":
        return grouped_count(data["N_data"], cfg["indices"])
    return weighted_group(data[metric], data["N_data"], cfg["indices"])


combined_monthly_rows = []
for cfg in EVALUATIONS:
    for run_name in run_order_for_cfg(cfg):
        data = runs[run_name]["monthly"]
        for metric in MONTHLY_METRICS:
            vals = grouped_metric(data, cfg, metric)
            if vals is None:
                continue
            for month, value in zip(months, vals):
                combined_monthly_rows.append(
                    {
                        "month": month.strftime("%Y-%m"),
                        "group": cfg["group"],
                        "run": run_name,
                        "run_label": RUN_LABELS[run_name],
                        "baseline": cfg["baseline"],
                        "metric": metric,
                        "metric_label": METRIC_LABELS[metric],
                        "value": value,
                    }
                )

combined_monthly_metrics = pd.DataFrame(combined_monthly_rows)
combined_monthly_path = OUT_DIR / "combined_monthly_all_metrics.csv"
combined_monthly_metrics.to_csv(combined_monthly_path, index=False)

combined_temporal_rows = []
for cfg in EVALUATIONS:
    for run_name in run_order_for_cfg(cfg):
        data = runs[run_name]["temporal"]
        for metric in TEMPORAL_METRICS:
            vals = grouped_metric(data, cfg, metric)
            if vals is None:
                continue
            combined_temporal_rows.append(
                {
                    "group": cfg["group"],
                    "run": run_name,
                    "run_label": RUN_LABELS[run_name],
                    "baseline": cfg["baseline"],
                    "metric": metric,
                    "metric_label": METRIC_LABELS[metric],
                    "mean": area_weighted_mean(vals),
                    "median": np.nanmedian(vals),
                    "std": area_weighted_std(vals),
                    "p05": np.nanpercentile(vals, 5),
                    "p95": np.nanpercentile(vals, 95),
                    "n_valid_tiles": int(np.isfinite(vals).sum()),
                }
            )

combined_temporal_metrics = pd.DataFrame(combined_temporal_rows)
combined_temporal_path = OUT_DIR / "combined_temporal_all_metrics_summary.csv"
combined_temporal_metrics.to_csv(combined_temporal_path, index=False)

combined_monthly_metrics.head(), combined_temporal_metrics.head()

## Fig. 7: all combined species-group monthly mean metrics

Monthly combined species-group mean diagnostics. Direction depends on the metric; these panels are for context rather than a single skill score.


In [ ]:
def plot_combined_metric_grid(metrics, stem, title):
    fig, axes = plt.subplots(len(metrics), len(EVALUATIONS), figsize=(15.2, 2.1 * len(metrics) + 1.0), sharex=True, constrained_layout=True)
    if len(metrics) == 1:
        axes = axes[np.newaxis, :]

    legend_handles = {}
    for row, metric in enumerate(metrics):
        for col, cfg in enumerate(EVALUATIONS):
            ax = axes[row, col]
            for run_name in visible_run_order(cfg):
                data = runs[run_name]["monthly"]
                vals = grouped_metric(data, cfg, metric)
                if vals is None or not np.isfinite(vals).any():
                    continue
                color = "0.25" if run_name == cfg["baseline"] else RUN_COLORS.get(run_name, "0.5")
                linestyle = "--" if run_name == cfg["baseline"] else "-"
                linewidth = 1.5 if run_name == cfg["baseline"] else 1.8
                marker = "o" if run_name == "DA_H121" else None
                markersize = 2.6 if run_name == "DA_H121" else 0
                markevery = 6 if run_name == "DA_H121" else None
                zorder = 5 if run_name == "DA_H121" else 2
                y = vals / 1e6 if metric == "N_data" else vals
                handle = ax.plot(
                    months,
                    y,
                    color=color,
                    linestyle=linestyle,
                    linewidth=linewidth,
                    marker=marker,
                    markersize=markersize,
                    markevery=markevery,
                    zorder=zorder,
                    label=RUN_LABELS[run_name],
                )[0]
                legend_handles[RUN_LABELS[run_name]] = handle
            ax.set_title(f"{cfg['group']} - {METRIC_LABELS[metric]}", loc="left", fontsize=10)
            ylabel = "million obs" if metric == "N_data" else "weighted species-group mean"
            ax.set_ylabel(ylabel)
            format_time_axis(ax)

    for ax in axes[-1, :]:
        ax.set_xlabel("Month")
    add_panel_labels(axes)
    fig.legend(list(legend_handles.values()), list(legend_handles.keys()), loc="center left", ncol=1, frameon=False, bbox_to_anchor=(1.01, 0.5))
    fig.suptitle(title, y=1.015, fontsize=13)
    savefig(fig, stem)


plot_combined_metric_grid(MEAN_METRICS, "fig07_combined_monthly_mean_metrics", "Combined species-group monthly mean metrics")

## Fig. 8: all combined species-group monthly stddev metrics

Monthly combined species-group spread diagnostics. Lower O-F and O-A stddev values generally indicate tighter fit to observations.


In [ ]:
plot_combined_metric_grid(STDV_METRICS, "fig08_combined_monthly_stdv_metrics", "Combined species-group monthly stddev metrics")

## Notes for later tweaks

- The combined species-group metrics use `N_data` as species weights and require at least `NMIN` observations for each species contribution; full-period tile means use tile-area weights.
- The maps are full-period aggregates because the available temporal files collapse the 72 months into one tile-level statistic.
- A DA run is always compared to the OL/background file containing the matching observation species group: SMAP uses `OL_vs_SMAPobs`, legacy ASCAT uses `OL_vs_legacyobs`, and H121 ASCAT uses `OL_vs_H121obs`.
- Added jointly matched OL legacy-vs-H121 ASCAT diagnostics use `OL_legacy_h121_xmask`, where legacy/H121 species pairs are sampled from the same OL run.
- The notebook writes PNG figures, displays them inline, and writes CSV summaries to `projects/ascat_da/output/omf_h121_legacy_figures`.
- Support F uses monthly global OL summaries; seasonal maps would require monthly-by-tile O summaries, which are not in this cache.
- Supports G-J use full-period OL monitor summaries to diagnose H121-minus-legacy observation and O-F differences.
